# Pipeline de Análise de Produção Acadêmica (Lattes + ORCID + Scopus + Eventos)

Este notebook consolida em um único pipeline a produção bibliográfica de
**três fontes** por professor — Lattes (JSONs brutos), ORCID (API pública) e
Scopus (`pybliometrics`) — cruza com a base de percentil Scopus e com a base
de eventos classificados, detecta coautoria de alunos, e persiste tudo em
`pesquisadores_teste.duckdb`.

## O que mudou em relação à versão anterior (só Lattes)

- **Extração por fonte**: cada fonte (Lattes/ORCID/Scopus) agora produz seus
  próprios DataFrames de artigos de periódico e de trabalhos de congresso,
  **no mesmo schema final** (mesmas colunas de antes, mais uma coluna
  `fonte`). Isso é o que permite simplesmente concatenar as três fontes.
- **Unificação com deduplicação**: como o mesmo artigo pode aparecer em mais
  de uma fonte (ex.: Lattes e Scopus), a união usa `doi` normalizado (e,
  na ausência de DOI, `título normalizado + id_lattes`) como chave de
  deduplicação — sem isso, os índices de produtividade do `app.py` contariam
  o mesmo artigo mais de uma vez.
- **Tratamentos rodam uma vez, sobre a base já unificada e deduplicada**: o
  cruzamento com a planilha de percentil Scopus, o cruzamento com a base de
  eventos e a detecção de coautoria de alunos passam a considerar
  `COALESCE(titulo_revista_lattes, titulo_revista_scopus)` como nome do
  periódico (e o mesmo princípio para eventos), então artigos vindos de
  qualquer fonte são tratados da mesma forma.
- **Propagação de volta**: depois de calculado, o resultado do tratamento é
  mesclado de volta tanto na tabela unificada quanto nas três tabelas por
  fonte (pré-deduplicação) — cada uma preservando a granularidade "uma linha
  por publicação por fonte".
- **Banco de dados final com 11 tabelas** em vez de 5: as 5 tabelas de
  sempre (`tb_professores`, `tb_alunos`, `tb_orientacoes`,
  `tb_artigo_periodico`, `tb_artigo_conferencia` — mesmo formato de antes,
  para não quebrar `app.py`) mais 6 tabelas novas, uma por fonte e tipo de
  produção (`tb_artigo_periodico_lattes/orcid/scopus`,
  `tb_artigo_conferencia_lattes/orcid/scopus`).

## Estrutura deste notebook

1. Configuração e imports
2. Extração dos arquivos JSON brutos (Lattes) e consolidação em DataFrames
3. Tratamento de `df_pessoas` (informações pessoais)
4. Tratamento de `df_orientacoes`
5. Tratamento de `df_bib_artigos`/`df_bib_trab_congresso` e reshape para o schema final (fonte Lattes)
6. Extração ORCID (fonte ORCID, mesmo schema final)
7. Extração Scopus (fonte Scopus, mesmo schema final)
8. Unificação das três fontes com deduplicação
9. Cruzamento de periódicos com a base de percentil Scopus (sobre a base unificada)
10. Cruzamento de trabalhos de congresso com a base de eventos classificados (sobre a base unificada)
11. Detecção de coautoria de alunos nas produções
12. Propagação do tratamento de volta para as tabelas por fonte
13. Persistência consolidada no banco DuckDB (11 tabelas)


## 1. Configuração e Imports

Todas as bibliotecas usadas em qualquer etapa do notebook são importadas uma
única vez aqui, incluindo as duas novas dependências de extração
(`orcid` e `pybliometrics`) e `python-dotenv` para ler credenciais do
arquivo `.env`.


In [1]:
import json
import re
import time
import unicodedata
import glob
from pathlib import Path

import numpy as np
import pandas as pd
import duckdb
import orcid
import pybliometrics
from pybliometrics.scopus import ScopusSearch
from rapidfuzz import fuzz
from dotenv import load_dotenv

load_dotenv()

# Caminho onde estão os currículos Lattes (em JSON) dos professores.
# Cada arquivo é o "dump" bruto de um currículo já raspado/processado.
CAMINHO_JSONS_PROFESSORES = 'dados_brutos/professores/ufrj/*.json'

# Caminho da pasta com os currículos Lattes (em JSON) dos alunos, usada
# na Seção 11 para detectar coautoria entre professores e alunos.
CAMINHO_PASTA_ALUNOS = 'dados_brutos/alunos'

# Planilha de percentis Scopus por periódico (usada na Seção 9).
ARQUIVO_PERCENTIL_SCOPUS = 'periodicos_percentil.xlsx'

# Base de eventos/conferências já classificados por estrato (usada na Seção 10).
ARQUIVO_EVENTOS_CLASSIFICADOS = 'eventos_classificados_dois_idiomas.csv'

# Lista de pessoas com id_lattes/orcid_id/scopus_author_id (usada nas Seções 6 e 7).
# Gerado a partir de 'ID_lattes - Página1.csv'; orcid_id/scopus_author_id
# começam vazios e precisam ser preenchidos manualmente.
ARQUIVO_LISTA_PESSOAS = 'lista_pessoas.csv'

# Arquivo DuckDB de destino final (usado na Seção 13).
# Mantido como um arquivo de teste por padrão -- troque para
# 'pesquisadores.duckdb' quando quiser promover o resultado para produção.
ARQUIVO_DUCKDB_DESTINO = 'pesquisadores_teste.duckdb'

print("OK: configuração e imports carregados.")

OK: configuração e imports carregados.


## 2. Extração dos JSONs Brutos (Lattes) e Consolidação em DataFrames

Cada arquivo JSON representa o currículo Lattes de **um professor**, já
estruturado em blocos (`informacoes_pessoais`, `bancas`, `eventos`,
`orientacoes`, `premios_titulos`, `projetos_pesquisa`, `producao_bibliografica`,
`producao_tecnica`, `patentes_registros`).

A estratégia é:

1. Percorrer todos os arquivos JSON da pasta de professores.
2. Para cada bloco de interesse, "achatar" (flatten) a estrutura em uma lista
   de dicionários e marcar cada registro com o `id_lattes` do professor de origem.
3. Acumular essas listas e, ao final, concatenar tudo em um único DataFrame
   por tipo de produção (ex.: todos os artigos de periódico de todos os
   professores em um só `df_bib_artigos`).

Esta seção é idêntica à versão anterior deste notebook — a extração do Lattes
continua sendo a base de tudo, agora é só uma das três fontes que alimentam o
pipeline final.


In [2]:
print("Localizando arquivos JSON de professores...")
caminhos_arquivos = glob.glob(CAMINHO_JSONS_PROFESSORES)

if not caminhos_arquivos:
    raise FileNotFoundError(
        f"Nenhum arquivo JSON encontrado em '{CAMINHO_JSONS_PROFESSORES}'. "
        "Verifique se o caminho está correto antes de continuar."
    )

print(f"{len(caminhos_arquivos)} arquivo(s) encontrado(s). Iniciando a extração...")

Localizando arquivos JSON de professores...
40 arquivo(s) encontrado(s). Iniciando a extração...


In [3]:
# ---------------------------------------------------------
# 2.1 Listas de acumulação — uma por tipo de produção/registro
# ---------------------------------------------------------
# Dados Gerais
lista_pessoas = []
lista_bancas = []
lista_eventos = []
lista_orientacoes = []
lista_premios = []
lista_projetos = []

# Produção Bibliográfica
lista_bib_artigos = []
lista_bib_livros = []
lista_bib_capitulos = []
lista_bib_trabalhos_congresso = []
lista_bib_resumos_expandidos = []
lista_bib_resumos_congresso = []
lista_bib_artigos_aceitos = []
lista_bib_apresentacoes = []
lista_bib_textos_jornais = []
lista_bib_outras = []

# Produção Técnica
lista_tec_softwares_patente = []
lista_tec_softwares_sem_patente = []
lista_tec_produtos = []
lista_tec_processos = []
lista_tec_trabalhos = []
lista_tec_outras = []
lista_tec_entrevistas = []

# Patentes e Registros
lista_pat_patentes = []
lista_pat_programas = []
lista_pat_desenhos = []

In [4]:
# ---------------------------------------------------------
# 2.2 Extração e achatamento (flatten) de cada arquivo JSON
# ---------------------------------------------------------
for arquivo in caminhos_arquivos:
    with open(arquivo, 'r', encoding='utf-8') as f:
        dados = json.load(f)

        id_lattes = dados.get('informacoes_pessoais', {}).get('id_lattes')
        if not id_lattes:
            # Sem id_lattes não há como vincular nenhum registro a um professor;
            # o arquivo é descartado.
            continue

        # --- INFORMAÇÕES PESSOAIS ---
        df_pessoa = pd.json_normalize(dados['informacoes_pessoais'])
        lista_pessoas.append(df_pessoa)

        # --- BANCAS (estrutura: {categoria: [itens]}) ---
        if 'bancas' in dados:
            for categoria, itens in dados['bancas'].items():
                if itens:
                    df_temp = pd.DataFrame(itens)
                    df_temp['id_lattes'] = id_lattes
                    df_temp['categoria_banca'] = categoria
                    if 'membros_banca' in df_temp.columns:
                        # Lista de membros é convertida para string para caber em uma célula tabular
                        df_temp['membros_banca'] = df_temp['membros_banca'].astype(str)
                    lista_bancas.append(df_temp)

        # --- EVENTOS PARTICIPADOS (estrutura: {categoria: [itens]}) ---
        if 'eventos' in dados:
            for categoria, itens in dados['eventos'].items():
                if itens:
                    df_temp = pd.DataFrame(itens)
                    df_temp['id_lattes'] = id_lattes
                    df_temp['categoria_evento'] = categoria
                    lista_eventos.append(df_temp)

        # --- ORIENTAÇÕES (estrutura: {status: {nivel: [itens]}}) ---
        if 'orientacoes' in dados:
            for status, dicionario_niveis in dados['orientacoes'].items():
                for nivel, itens in dicionario_niveis.items():
                    if itens:
                        df_temp = pd.DataFrame(itens)
                        df_temp['id_lattes'] = id_lattes
                        df_temp['status'] = status
                        df_temp['nivel'] = nivel
                        lista_orientacoes.append(df_temp)

        # --- PRÊMIOS E TÍTULOS (lista simples) ---
        if 'premios_titulos' in dados and dados['premios_titulos']:
            df_temp = pd.DataFrame(dados['premios_titulos'])
            df_temp['id_lattes'] = id_lattes
            lista_premios.append(df_temp)

        # --- PROJETOS DE PESQUISA (lista simples, com campos longos) ---
        if 'projetos_pesquisa' in dados and dados['projetos_pesquisa']:
            df_temp = pd.DataFrame(dados['projetos_pesquisa'])
            df_temp['id_lattes'] = id_lattes
            for col in ['descricao', 'integrantes', 'financiadores']:
                if col in df_temp.columns:
                    df_temp[col] = df_temp[col].astype(str)
            lista_projetos.append(df_temp)

        # --- PRODUÇÃO BIBLIOGRÁFICA (estrutura: {chave: [itens]}) ---
        prod_bib = dados.get('producao_bibliografica', {})

        def add_to_list(chave, lista_destino):
            """Extrai uma chave de produção bibliográfica e empilha na lista destino."""
            itens = prod_bib.get(chave, [])
            if itens:
                df_temp = pd.DataFrame(itens)
                df_temp['id_lattes'] = id_lattes
                lista_destino.append(df_temp)

        add_to_list('artigos_periodicos', lista_bib_artigos)
        add_to_list('livros_publicados', lista_bib_livros)
        add_to_list('capitulos_livros', lista_bib_capitulos)
        add_to_list('trabalhos_completos_congressos', lista_bib_trabalhos_congresso)
        add_to_list('resumos_expandidos', lista_bib_resumos_expandidos)
        add_to_list('resumos_congressos', lista_bib_resumos_congresso)
        add_to_list('artigos_aceitos', lista_bib_artigos_aceitos)
        add_to_list('apresentacoes_trabalhos', lista_bib_apresentacoes)
        add_to_list('textos_jornais', lista_bib_textos_jornais)
        add_to_list('outras_producoes', lista_bib_outras)

        # --- PRODUÇÃO TÉCNICA (estrutura: {chave: [itens]}) ---
        prod_tec = dados.get('producao_tecnica', {})

        def add_to_list_tec(chave, lista_destino):
            """Extrai uma chave de produção técnica e empilha na lista destino."""
            itens = prod_tec.get(chave, [])
            if itens:
                df_temp = pd.DataFrame(itens)
                df_temp['id_lattes'] = id_lattes
                lista_destino.append(df_temp)

        add_to_list_tec('softwares_com_patente', lista_tec_softwares_patente)
        add_to_list_tec('softwares_sem_patente', lista_tec_softwares_sem_patente)
        add_to_list_tec('produtos_tecnologicos', lista_tec_produtos)
        add_to_list_tec('processos_tecnicas', lista_tec_processos)
        add_to_list_tec('trabalhos_tecnicos', lista_tec_trabalhos)
        add_to_list_tec('outras_producoes_tecnicas', lista_tec_outras)
        add_to_list_tec('entrevistas', lista_tec_entrevistas)

        # --- PATENTES E REGISTROS (estrutura: {chave: [itens]}) ---
        patentes = dados.get('patentes_registros', {})

        def add_to_list_pat(chave, lista_destino):
            """Extrai uma chave de patentes/registros e empilha na lista destino."""
            itens = patentes.get(chave, [])
            if itens:
                df_temp = pd.DataFrame(itens)
                df_temp['id_lattes'] = id_lattes
                lista_destino.append(df_temp)

        add_to_list_pat('patentes', lista_pat_patentes)
        add_to_list_pat('programas_computador', lista_pat_programas)
        add_to_list_pat('desenhos_industriais', lista_pat_desenhos)

print("Extração e achatamento concluídos para todos os arquivos.")

Extração e achatamento concluídos para todos os arquivos.


In [5]:
# ---------------------------------------------------------
# 2.3 Consolidação: cada lista de DataFrames parciais (um por professor)
#     é concatenada em um único DataFrame final por tipo de produção.
# ---------------------------------------------------------
def consolidar(lista):
    """Concatena uma lista de DataFrames parciais; retorna DataFrame vazio se a lista estiver vazia."""
    return pd.concat(lista, ignore_index=True) if lista else pd.DataFrame()

# DataFrames Gerais
df_pessoas = consolidar(lista_pessoas)
df_bancas = consolidar(lista_bancas)
df_eventos = consolidar(lista_eventos)
df_orientacoes = consolidar(lista_orientacoes)
df_premios = consolidar(lista_premios)
df_projetos = consolidar(lista_projetos)

# DataFrames de Produção Bibliográfica
df_bib_artigos = consolidar(lista_bib_artigos)
df_bib_livros = consolidar(lista_bib_livros)
df_bib_capitulos = consolidar(lista_bib_capitulos)
df_bib_trab_congresso = consolidar(lista_bib_trabalhos_congresso)
df_bib_resumos_exp = consolidar(lista_bib_resumos_expandidos)
df_bib_resumos_cong = consolidar(lista_bib_resumos_congresso)
df_bib_art_aceitos = consolidar(lista_bib_artigos_aceitos)
df_bib_apresentacoes = consolidar(lista_bib_apresentacoes)
df_bib_textos_jornais = consolidar(lista_bib_textos_jornais)
df_bib_outras = consolidar(lista_bib_outras)

# DataFrames de Produção Técnica
df_tec_soft_patente = consolidar(lista_tec_softwares_patente)
df_tec_soft_sem_patente = consolidar(lista_tec_softwares_sem_patente)
df_tec_produtos = consolidar(lista_tec_produtos)
df_tec_processos = consolidar(lista_tec_processos)
df_tec_trabalhos = consolidar(lista_tec_trabalhos)
df_tec_outras = consolidar(lista_tec_outras)
df_tec_entrevistas = consolidar(lista_tec_entrevistas)

# DataFrames de Patentes e Registros
df_pat_patentes = consolidar(lista_pat_patentes)
df_pat_programas = consolidar(lista_pat_programas)
df_pat_desenhos = consolidar(lista_pat_desenhos)

print("DataFrames consolidados. Resumo de volumes:")
print(f"  Professores (df_pessoas):              {len(df_pessoas)}")
print(f"  Orientações (df_orientacoes):           {len(df_orientacoes)}")
print(f"  Artigos de periódico (df_bib_artigos):  {len(df_bib_artigos)}")
print(f"  Trabalhos de congresso (df_bib_trab_congresso): {len(df_bib_trab_congresso)}")

DataFrames consolidados. Resumo de volumes:
  Professores (df_pessoas):              40
  Orientações (df_orientacoes):           2907
  Artigos de periódico (df_bib_artigos):  2018
  Trabalhos de congresso (df_bib_trab_congresso): 3670


## 3. Tratamento de `df_pessoas` (Informações Pessoais)

Limpeza padrão de cadastro: strings vazias→nulo, datas, remoção de marcador
"*" no rótulo, tipagem da chave primária e do texto de resumo. Ao final,
mesclamos `orcid_id`/`scopus_author_id` de `lista_pessoas.csv` — esses dois
IDs só existem para consultar as APIs externas (Seções 6 e 7), mas também
são guardados em `tb_professores` para rastreabilidade.


In [6]:
# 3.1 Substitui strings vazias ou só com espaços por NaN (nulo real)
df_pessoas.replace(r'^\s*$', np.nan, regex=True, inplace=True)

# 3.2 Converte a data de atualização do CV ('15/10/2025') para datetime.
#     errors='coerce' faz datas inválidas virarem nulo em vez de quebrar o script.
if 'atualizacao_cv' in df_pessoas.columns:
    df_pessoas['atualizacao_cv'] = pd.to_datetime(
        df_pessoas['atualizacao_cv'],
        format='%d/%m/%Y',
        errors='coerce'
    )

# 3.3 Limpeza do campo 'rotulo': remove o asterisco e espaços, e transforma
#     o texto literal "Sem rótulo" em nulo verdadeiro.
if 'rotulo' in df_pessoas.columns:
    df_pessoas['rotulo'] = df_pessoas['rotulo'].str.replace('*', '', regex=False).str.strip()
    df_pessoas['rotulo'] = df_pessoas['rotulo'].replace('Sem rótulo', np.nan)

# 3.4 Garante que a chave primária (id_lattes) seja sempre string,
#     evitando inconsistências de tipo em merges/joins posteriores.
df_pessoas['id_lattes'] = df_pessoas['id_lattes'].astype(str)

# 3.5 Remove espaços/quebras de linha nas bordas do texto de resumo do CV.
if 'texto_resumo' in df_pessoas.columns:
    df_pessoas['texto_resumo'] = df_pessoas['texto_resumo'].str.strip()

# 3.6 Mescla orcid_id/scopus_author_id da lista de mapeamento de pessoas.
#     Sem essa lista o cadastro em tb_professores não quebra -- as duas
#     colunas simplesmente ficam nulas.
if Path(ARQUIVO_LISTA_PESSOAS).exists():
    df_lista_pessoas = pd.read_csv(ARQUIVO_LISTA_PESSOAS, dtype=str)
    df_lista_pessoas['id_lattes'] = df_lista_pessoas['id_lattes'].astype(str).str.strip()
    df_lista_pessoas['orcid_id'] = df_lista_pessoas['orcid_id'].str.strip().replace('', np.nan)
    df_lista_pessoas['scopus_author_id'] = df_lista_pessoas['scopus_author_id'].str.strip().replace('', np.nan)

    df_pessoas = df_pessoas.merge(
        df_lista_pessoas[['id_lattes', 'orcid_id', 'scopus_author_id']],
        on='id_lattes', how='left'
    )
else:
    print(f"AVISO: '{ARQUIVO_LISTA_PESSOAS}' não encontrado -- orcid_id/scopus_author_id ficarão nulos.")
    df_pessoas['orcid_id'] = pd.NA
    df_pessoas['scopus_author_id'] = pd.NA

print("Tratamento de df_pessoas concluído.")
df_pessoas.info()

Tratamento de df_pessoas concluído.
<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 13 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id_lattes              40 non-null     str           
 1   nome_completo          40 non-null     str           
 2   nome_citacoes          40 non-null     str           
 3   sexo                   40 non-null     str           
 4   rotulo                 0 non-null      str           
 5   periodo                0 non-null      str           
 6   bolsa_produtividade    21 non-null     str           
 7   endereco_profissional  39 non-null     str           
 8   atualizacao_cv         40 non-null     datetime64[us]
 9   url                    40 non-null     str           
 10  texto_resumo           40 non-null     str           
 11  orcid_id               31 non-null     str           
 12  scopus_author_id       30 non-null     st

## 4. Tratamento de `df_orientacoes`

**Atenção à ordem**: a coluna original `titulo` é renomeada para
`titulo_trabalho` *antes* da limpeza de texto em lote, porque a limpeza já
referencia o nome novo (`titulo_trabalho`).

> **Nota de atenção (herdada do notebook original):** a Seção 13 insere
> `df_orientacoes` no banco esperando uma coluna `ano_inicio`. Essa coluna
> nunca é criada nem renomeada em nenhuma etapa de tratamento — ela só existe
> se o JSON bruto de orientações já trouxer um campo chamado literalmente
> `ano_inicio`. Se o seu JSON de origem usa outro nome para o ano de início da
> orientação, adicione aqui um `rename` (no mesmo padrão da linha abaixo que
> renomeia `titulo` → `titulo_trabalho`) antes de chegar à Seção 13, ou a
> inserção no banco falhará com `KeyError`/coluna inexistente.


In [7]:
# 4.1 Renomeia 'titulo' para 'titulo_trabalho' (nome mais descritivo,
#     usado pela limpeza de texto na sequência e pelo schema do banco).
df_orientacoes = df_orientacoes.rename(columns={'titulo': 'titulo_trabalho'})

print("Tratamento da tabela de orientações...")

# 4.2 Limpeza de texto: remove espaços duplos/quebras de linha escondidas e
#     preenche vazios com um rótulo explícito em vez de deixá-los como string vazia.
colunas_texto = ['titulo_trabalho', 'orientando', 'tipo_trabalho', 'instituicao', 'curso']
for col in colunas_texto:
    df_orientacoes[col] = df_orientacoes[col].astype(str).str.strip()
    df_orientacoes[col] = df_orientacoes[col].replace(
        {'': 'Não informado', 'nan': 'Não informado', 'None': 'Não informado'}
    )

# 4.3 Conversão segura do ano de conclusão (float -> Int64, que aceita nulos).
df_orientacoes['ano_conclusao'] = df_orientacoes['ano_conclusao'].astype('Int64')

# 4.4 Padroniza os valores de 'nivel' para rótulos amigáveis (usados em gráficos).
mapeamento_nivel = {
    'mestrado': 'Mestrado',
    'doutorado': 'Doutorado',
    'tcc': 'TCC',
    'iniciacao_cientifica': 'Iniciação Científica',
    'pos_doutorado': 'Pós-Doutorado',
    'especializacao': 'Especialização',
    'outros': 'Outros'
}
df_orientacoes['nivel'] = df_orientacoes['nivel'].map(mapeamento_nivel).fillna(df_orientacoes['nivel'])

# 4.5 Padroniza os valores de 'status' para rótulos amigáveis.
mapeamento_status = {
    'concluidas': 'Concluída',
    'em_andamento': 'Em Andamento'
}
df_orientacoes['status'] = df_orientacoes['status'].map(mapeamento_status).fillna(df_orientacoes['status'])

print("Tratamento concluído. Amostra dos dados tratados:")
display(df_orientacoes[['orientando', 'nivel', 'status', 'ano_conclusao']].head())

Tratamento da tabela de orientações...
Tratamento concluído. Amostra dos dados tratados:


,orientando,nivel,status,ano_conclusao
0,Rodrigo Fernandes Souto,Doutorado,Em Andamento,<NA>
1,Bruno Bandeira Monteiro,Doutorado,Em Andamento,<NA>
2,Rafael Paladini Meirelles,Mestrado,Em Andamento,<NA>
3,Eduardo Naslausky,Mestrado,Em Andamento,<NA>
4,Caio de Campos,TCC,Em Andamento,<NA>


## 5. Tratamento de `df_bib_artigos`/`df_bib_trab_congresso` e Reshape (fonte Lattes)

Antes de qualquer cruzamento com bases externas, os dois DataFrames de
produção bibliográfica do Lattes recebem uma limpeza básica: nulos reais,
padronização de maiúsculas no nome do periódico/evento, tipagem de ano e
remoção de espaços ocultos em texto (idêntico à versão anterior deste
notebook).

Em seguida, os dois DataFrames são reorganizados (*reshape*) para o **schema
final** — as mesmas colunas usadas por `tb_artigo_periodico` e
`tb_artigo_conferencia` — com os campos que dependem de cruzamento (percentil,
estrato, `match_adequado`, `coautoria_aluno` etc.) começando nulos. Essa é a
mesma estrutura que as extrações de ORCID e Scopus (Seções 6 e 7) também vão
produzir, o que torna a união das três fontes (Seção 8) um simples `concat`.


In [8]:
print("Aplicando tratamentos na tabela 'df_bib_artigos'...")

if not df_bib_artigos.empty:

    # 1. Strings vazias/só espaços -> nulo real
    df_bib_artigos.replace(r'^\s*$', np.nan, regex=True, inplace=True)

    # 2. Nome da revista em maiúsculas e sem espaços nas bordas
    #    (necessário para o cruzamento exato com a base Scopus na Seção 9)
    if 'revista' in df_bib_artigos.columns:
        df_bib_artigos['revista'] = df_bib_artigos['revista'].str.upper().str.strip()

    # 3. Ano como inteiro com suporte a nulo (Int64); valores inválidos -> nulo
    if 'ano' in df_bib_artigos.columns:
        df_bib_artigos['ano'] = pd.to_numeric(df_bib_artigos['ano'], errors='coerce').astype('Int64')

    # 4. Remove espaços ocultos nas colunas de texto livre
    colunas_texto = ['titulo', 'doi', 'issn', 'volume', 'numero', 'paginas']
    for col in colunas_texto:
        if col in df_bib_artigos.columns:
            df_bib_artigos[col] = df_bib_artigos[col].str.strip()

    # 5. Garante tipagem string na chave primária
    df_bib_artigos['id_lattes'] = df_bib_artigos['id_lattes'].astype(str)

print("Tratamento de 'df_bib_artigos' concluído.")
display(df_bib_artigos[['ano', 'revista', 'doi', 'issn']].head())

Aplicando tratamentos na tabela 'df_bib_artigos'...
Tratamento de 'df_bib_artigos' concluído.


,ano,revista,doi,issn
0,2021,HISTORY AND PHILOSOPHY OF LOGIC,http://dx.doi.org/10.1080/01445340.2021.1971005,1464-5149
1,2021,DISCRETE APPLIED MATHEMATICS,http://dx.doi.org/10.1016/j.dam.2020.09.007,0166-218X
2,2020,DISCRETE MATHEMATICS,http://dx.doi.org/10.1016/j.disc.2019.111717,0012-365X
3,2020,DISCRETE APPLIED MATHEMATICS,http://dx.doi.org/10.1016/j.dam.2019.03.022,0166-218X
4,2019,ELECTRONIC NOTES IN THEORETICAL COMPUTER SCIENCE,http://dx.doi.org/10.1016/j.entcs.2019.08.027,1571-0661


In [9]:
print("Aplicando tratamentos na tabela 'df_bib_trab_congresso'...")

if not df_bib_trab_congresso.empty:

    # 1. Strings vazias/só espaços -> nulo real
    df_bib_trab_congresso.replace(r'^\s*$', np.nan, regex=True, inplace=True)

    # 2. Nome do evento em maiúsculas e sem espaços nas bordas
    #    (necessário para o cruzamento com a base de eventos na Seção 10)
    if 'evento' in df_bib_trab_congresso.columns:
        df_bib_trab_congresso['evento'] = df_bib_trab_congresso['evento'].str.upper().str.strip()

    # 3. Ano como inteiro com suporte a nulo (Int64)
    if 'ano' in df_bib_trab_congresso.columns:
        df_bib_trab_congresso['ano'] = pd.to_numeric(df_bib_trab_congresso['ano'], errors='coerce').astype('Int64')

    # 4. Remove espaços ocultos nas colunas de texto livre
    colunas_texto = ['titulo', 'doi', 'isbn', 'paginas']
    for col in colunas_texto:
        if col in df_bib_trab_congresso.columns:
            df_bib_trab_congresso[col] = df_bib_trab_congresso[col].str.strip()

    # 5. Padronização da coluna 'autores': separador único (vírgula),
    #    sem espaços duplicados, e em maiúsculas (facilita buscas futuras,
    #    inclusive a detecção de coautoria de alunos na Seção 11).
    if 'autores' in df_bib_trab_congresso.columns:
        df_bib_trab_congresso['autores'] = df_bib_trab_congresso['autores'].str.strip()
        df_bib_trab_congresso['autores'] = df_bib_trab_congresso['autores'].str.replace(';', ',', regex=False)
        df_bib_trab_congresso['autores'] = df_bib_trab_congresso['autores'].str.replace(r'\s+', ' ', regex=True)
        df_bib_trab_congresso['autores'] = df_bib_trab_congresso['autores'].str.upper()

    # 6. Garante tipagem string na chave primária
    df_bib_trab_congresso['id_lattes'] = df_bib_trab_congresso['id_lattes'].astype(str)

print("Tratamento de 'df_bib_trab_congresso' concluído.")
display(df_bib_trab_congresso[['ano', 'evento', 'autores']].head())

Aplicando tratamentos na tabela 'df_bib_trab_congresso'...
Tratamento de 'df_bib_trab_congresso' concluído.


,ano,evento,autores
0,2022,WORKSHOP BRASILEIRO DE LÓGICA,"CERIOLI, MÁRCIA R., FREITAS, RENATA DE , VIANA..."
1,2021,DIAGRAMS,"CERIOLI, M. R., SUGUITANI, L. , VIANA, PETRUCIO"
2,2017,LATIN AND AMERICAN ALGORITHMS,"CERIOLI, M. R., FERNANDES, C. G. , GOMES, R. ,..."
3,2017,CNMAC 2016 XXXVI CONGRESSO NACIONAL DE MATEMÁT...,"CERIOLI, MA'RCIA, NOBREGA, HUGO , SILVEIRA, GU..."
4,2015,XXXV CNMAC CONGRESSO NACIONAL DE MATEMÁTICA AP...,"BARROS, GABRIEL F. , POSNER, DANIEL F. D. , CE..."


### 5.1 Reshape para o schema final

`COLUNAS_PERIODICO`/`COLUNAS_CONGRESSO` definem o schema final único —
usado pelas três fontes (Lattes, ORCID, Scopus) e, mais tarde, pela tabela
unificada. Os campos que ainda dependem de cruzamento ficam nulos aqui;
são preenchidos nas Seções 9-11 e propagados de volta na Seção 12.


In [10]:
COLUNAS_PERIODICO = [
    'id_lattes', 'titulo_artigo', 'titulo_revista_lattes', 'ano_pub', 'doi',
    'autores', 'match_adequado', 'coautoria_aluno', 'id_scopus',
    'titulo_revista_scopus', 'maior_percentil', 'codigo_area_maior_percentil',
    'area_maior_percentil', 'issn', 'computation_area', 'fonte',
]

COLUNAS_CONGRESSO = [
    'id_lattes', 'titulo_artigo', 'ano', 'doi', 'autores',
    'titulo_evento_lattes', 'paginas', 'sigla_evento_google',
    'titulo_evento_google', 'estrato', 'tipo_match', 'coautoria_aluno', 'fonte',
]


def montar_df_vazio(colunas):
    """Cria um DataFrame vazio já com as colunas do schema final, evitando
    KeyError mais adiante quando uma fonte não retorna nenhum registro."""
    return pd.DataFrame(columns=colunas)


print("Reorganizando 'df_bib_artigos' para o schema final (fonte LATTES)...")
if not df_bib_artigos.empty:
    df_artigos_periodico_lattes = pd.DataFrame({
        'id_lattes': df_bib_artigos['id_lattes'],
        'titulo_artigo': df_bib_artigos.get('titulo'),
        'titulo_revista_lattes': df_bib_artigos.get('revista'),
        'ano_pub': df_bib_artigos.get('ano'),
        'doi': df_bib_artigos.get('doi'),
        'autores': df_bib_artigos.get('autores'),
        'match_adequado': pd.NA,
        'coautoria_aluno': pd.NA,
        'id_scopus': pd.NA,
        'titulo_revista_scopus': pd.NA,
        'maior_percentil': pd.NA,
        'codigo_area_maior_percentil': pd.NA,
        'area_maior_percentil': pd.NA,
        'issn': df_bib_artigos.get('issn'),
        'computation_area': pd.NA,
        'fonte': 'LATTES',
    })[COLUNAS_PERIODICO]
else:
    df_artigos_periodico_lattes = montar_df_vazio(COLUNAS_PERIODICO)

print("Reorganizando 'df_bib_trab_congresso' para o schema final (fonte LATTES)...")
if not df_bib_trab_congresso.empty:
    df_artigos_congresso_lattes = pd.DataFrame({
        'id_lattes': df_bib_trab_congresso['id_lattes'],
        'titulo_artigo': df_bib_trab_congresso.get('titulo'),
        'ano': df_bib_trab_congresso.get('ano'),
        'doi': df_bib_trab_congresso.get('doi'),
        'autores': df_bib_trab_congresso.get('autores'),
        'titulo_evento_lattes': df_bib_trab_congresso.get('evento'),
        'paginas': df_bib_trab_congresso.get('paginas'),
        'sigla_evento_google': pd.NA,
        'titulo_evento_google': pd.NA,
        'estrato': pd.NA,
        'tipo_match': pd.NA,
        'coautoria_aluno': pd.NA,
        'fonte': 'LATTES',
    })[COLUNAS_CONGRESSO]
else:
    df_artigos_congresso_lattes = montar_df_vazio(COLUNAS_CONGRESSO)

print(f"df_artigos_periodico_lattes: {len(df_artigos_periodico_lattes)} linhas")
print(f"df_artigos_congresso_lattes: {len(df_artigos_congresso_lattes)} linhas")
display(df_artigos_periodico_lattes.head())

Reorganizando 'df_bib_artigos' para o schema final (fonte LATTES)...
Reorganizando 'df_bib_trab_congresso' para o schema final (fonte LATTES)...
df_artigos_periodico_lattes: 2018 linhas
df_artigos_congresso_lattes: 3670 linhas


,id_lattes,titulo_artigo,titulo_revista_lattes,ano_pub,doi,autores,match_adequado,coautoria_aluno,id_scopus,titulo_revista_scopus,maior_percentil,codigo_area_maior_percentil,area_maior_percentil,issn,computation_area,fonte
0,0211300683784278,On the (In)Dependence of the Peano Axioms for ...,HISTORY AND PHILOSOPHY OF LOGIC,2021,http://dx.doi.org/10.1080/01445340.2021.1971005,"CERIOLI, MÁRCIA R.; NOBREGA, HUGO ; SILVEIRA, ...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1464-5149,<NA>,LATTES
1,0211300683784278,Short proofs on the structure of general parti...,DISCRETE APPLIED MATHEMATICS,2021,http://dx.doi.org/10.1016/j.dam.2020.09.007,"CERIOLI, MÁRCIA R.; MARTINS, TAÍSA",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0166-218X,<NA>,LATTES
2,0211300683784278,Transversals of longest paths,DISCRETE MATHEMATICS,2020,http://dx.doi.org/10.1016/j.disc.2019.111717,"CERIOLI, MÁRCIA R.; FERNANDES, CRISTINA G. ; G...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0012-365X,<NA>,LATTES
3,0211300683784278,Intersection of longest paths in graph classes,DISCRETE APPLIED MATHEMATICS,2020,http://dx.doi.org/10.1016/j.dam.2019.03.022,"CERIOLI, MÁRCIA R.; LIMA, PALOMA T.",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0166-218X,<NA>,LATTES
4,0211300683784278,On Edge-magic Labelings of Forests,ELECTRONIC NOTES IN THEORETICAL COMPUTER SCIENCE,2019,http://dx.doi.org/10.1016/j.entcs.2019.08.027,"CERIOLI, M. R.; FERNANDES, C. G. ; LEE, O. ; L...",<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1571-0661,<NA>,LATTES


## 6. Extração ORCID (fonte ORCID)

Adaptado de `orcid_1.ipynb`. Para cada pessoa em `lista_pessoas.csv` que já
tenha um `orcid_id` preenchido, lemos o registro público completo do ORCID e
extraímos a lista de `works` (trabalhos), classificando cada um em
**periódico** ou **congresso/evento** a partir do campo `type` retornado pela
API. O resultado já sai no schema final (`COLUNAS_PERIODICO`/
`COLUNAS_CONGRESSO`), com `fonte='ORCID'` e os campos de cruzamento nulos.

> **Limitação conhecida:** a API pública do ORCID não retorna o nome dos
> autores no resumo de cada trabalho (`work-summary`) na maioria dos casos —
> só o `credit-name` de quem cadastrou a publicação, quando preenchido. Por
> isso `autores` tende a ficar nulo para a maior parte das linhas de origem
> ORCID (mesma limitação já observada em `orcid_1.ipynb`; corrigir isso
> exigiria 1 requisição extra por trabalho e está fora do escopo desta
> mudança).


In [11]:
CLIENT_ID = __import__('os').getenv('ORCID_CLIENT_ID')
CLIENT_SECRET = __import__('os').getenv('ORCID_CLIENT_SECRET')

df_artigos_periodico_orcid = montar_df_vazio(COLUNAS_PERIODICO)
df_artigos_congresso_orcid = montar_df_vazio(COLUNAS_CONGRESSO)

if not CLIENT_ID or not CLIENT_SECRET:
    print("AVISO: ORCID_CLIENT_ID/ORCID_CLIENT_SECRET não configurados no .env -- "
          "extração ORCID pulada (df_artigos_periodico_orcid/df_artigos_congresso_orcid ficam vazios).")
else:
    api_orcid = orcid.PublicAPI(CLIENT_ID, CLIENT_SECRET, sandbox=False)
    token_orcid = api_orcid.get_search_token_from_orcid()
    print("Autenticação ORCID realizada com sucesso!")

Autenticação ORCID realizada com sucesso!


In [12]:
# Tipos de trabalho do ORCID que consideramos "artigo de periódico"
# (https://info.orcid.org/documentation/integration-guide/orcid-work-types/)
TIPOS_PERIODICO_ORCID = {
    'journal_article',
}

# Tipos de trabalho do ORCID que consideramos "trabalho de congresso/conferência"
TIPOS_CONGRESSO_ORCID = {
    'conference_paper',
}


def extrair_autores_orcid(trabalho):
    """Extrai os nomes de autores/contribuidores de um work-summary do ORCID,
    juntando-os em uma única string separada por vírgula."""
    contribuidores = (trabalho.get('contributors') or {}).get('contributor', [])
    nomes = []
    for contrib in contribuidores:
        credit_name = (contrib.get('credit-name') or {})
        nome = credit_name.get('value') if credit_name else None
        if nome:
            nomes.append(nome.strip())
    return ', '.join(nomes) if nomes else pd.NA


def extrair_doi_orcid(trabalho):
    ext_ids_container = trabalho.get('external-ids') or {}
    ext_ids = ext_ids_container.get('external-id', []) if ext_ids_container else []
    for ident in ext_ids:
        if ident.get('external-id-type') == 'doi':
            return ident.get('external-id-value')
    return pd.NA


if CLIENT_ID and CLIENT_SECRET:
    df_lista_pessoas_orcid = pd.read_csv(ARQUIVO_LISTA_PESSOAS, dtype=str) if Path(ARQUIVO_LISTA_PESSOAS).exists() else pd.DataFrame(columns=['id_lattes', 'orcid_id'])
    df_lista_pessoas_orcid = df_lista_pessoas_orcid[df_lista_pessoas_orcid['orcid_id'].notna() & (df_lista_pessoas_orcid['orcid_id'].str.strip() != '')]

    lista_artigos_periodico_orcid = []
    lista_artigos_congresso_orcid = []

    print(f"Total de pessoas com orcid_id preenchido: {len(df_lista_pessoas_orcid)}")

    for _, pessoa in df_lista_pessoas_orcid.iterrows():
        id_lattes = str(pessoa['id_lattes']).strip()
        orcid_id = str(pessoa['orcid_id']).strip()

        print(f"Processando ORCID {orcid_id} (id_lattes={id_lattes})...")

        try:
            perfil = api_orcid.read_record_public(orcid_id, 'record', token_orcid)
        except Exception as exc:
            print(f"  -> Falha ao ler perfil de {orcid_id}: {exc}")
            continue

        atividades = perfil.get('activities-summary', {})
        grupos_trabalhos = atividades.get('works', {}).get('group', [])

        for grupo in grupos_trabalhos:
            trabalho = grupo.get('work-summary', [{}])[0]

            titulo = trabalho.get('title', {}).get('title', {}).get('value', pd.NA)
            tipo = (trabalho.get('type') or '').lower()

            pub_date = trabalho.get('publication-date') or {}
            ano_raw = pub_date.get('year', {}).get('value') if pub_date else None
            ano = pd.NA if not ano_raw else int(ano_raw)

            doi = extrair_doi_orcid(trabalho)
            autores = extrair_autores_orcid(trabalho)

            if tipo in TIPOS_CONGRESSO_ORCID:
                evento_dict = trabalho.get('journal-title') or {}
                lista_artigos_congresso_orcid.append({
                    'id_lattes': id_lattes, 'titulo_artigo': titulo, 'ano': ano, 'doi': doi,
                    'autores': autores, 'titulo_evento_lattes': evento_dict.get('value', pd.NA),
                    'paginas': pd.NA, 'sigla_evento_google': pd.NA, 'titulo_evento_google': pd.NA,
                    'estrato': pd.NA, 'tipo_match': pd.NA, 'coautoria_aluno': pd.NA, 'fonte': 'ORCID',
                })
            elif tipo in TIPOS_PERIODICO_ORCID:
                revista_dict = trabalho.get('journal-title') or {}
                lista_artigos_periodico_orcid.append({
                    'id_lattes': id_lattes, 'titulo_artigo': titulo,
                    'titulo_revista_lattes': revista_dict.get('value', pd.NA), 'ano_pub': ano,
                    'doi': doi, 'autores': autores, 'match_adequado': pd.NA, 'coautoria_aluno': pd.NA,
                    'id_scopus': pd.NA, 'titulo_revista_scopus': pd.NA, 'maior_percentil': pd.NA,
                    'codigo_area_maior_percentil': pd.NA, 'area_maior_percentil': pd.NA,
                    'issn': pd.NA, 'computation_area': pd.NA, 'fonte': 'ORCID',
                })

        time.sleep(0.2)  # respeita o rate-limit público da API do ORCID

    if lista_artigos_periodico_orcid:
        df_artigos_periodico_orcid = pd.DataFrame(lista_artigos_periodico_orcid)[COLUNAS_PERIODICO]
    if lista_artigos_congresso_orcid:
        df_artigos_congresso_orcid = pd.DataFrame(lista_artigos_congresso_orcid)[COLUNAS_CONGRESSO]

    print(f"\nArtigos de periódico extraídos do ORCID: {len(df_artigos_periodico_orcid)}")
    print(f"Trabalhos de congresso extraídos do ORCID: {len(df_artigos_congresso_orcid)}")

Total de pessoas com orcid_id preenchido: 31
Processando ORCID 0000-0002-2941-2522 (id_lattes=8386103364234098)...
Processando ORCID 0000-0002-6393-0876 (id_lattes=3957046121364560)...
Processando ORCID 0000-0001-9712-1930 (id_lattes=5727472788265998)...
Processando ORCID 0000-0002-7538-7305 (id_lattes=2002515486942024)...
Processando ORCID 0000-0002-9712-3140 (id_lattes=0211300683784278)...
Processando ORCID 0000-0003-4262-7242 (id_lattes=7816511618426042)...
Processando ORCID 0000-0002-4942-3624 (id_lattes=9237788190989316)...
Processando ORCID 0000-0002-2411-3101 (id_lattes=9063837162469343)...
Processando ORCID 0000-0003-3975-9076 (id_lattes=4783565791787812)...
Processando ORCID 0000-0001-9150-229X (id_lattes=3937502490683382)...
Processando ORCID 0000-0001-5080-1955 (id_lattes=8130520066599912)...
Processando ORCID 0000-0002-0870-3371 (id_lattes=1420784392366957)...
Processando ORCID 0000-0002-4231-9621 (id_lattes=9719247117370600)...
Processando ORCID 0000-0002-1927-7398 (id_lat

## 7. Extração Scopus (fonte Scopus)

Adaptado de `scopus_2.ipynb` (`pybliometrics`). Para cada pessoa em
`lista_pessoas.csv` com `scopus_author_id` preenchido, buscamos todas as
publicações indexadas via `ScopusSearch('AU-ID(...)')` e classificamos cada
uma em periódico ou congresso a partir de `subtype` — `ar` (Article) vira
periódico, `cp` (Conference Paper) vira congresso e todo o resto (`le`
Letter, `ed` Editorial, `no` Note, `er` Erratum, `re` Review, ...) é
descartado. `aggregationType` **não** serve para isso: ele diz só em que
veículo a publicação saiu, marcando "Journal" tanto para um artigo quanto
para uma carta ao editor publicada na mesma revista.

Além dos dois DataFrames no schema final, a extração monta
`df_catalogo_scopus` com **todas** as publicações retornadas pela API,
inclusive as descartadas e com seus `subtype`/`subtypeDescription`. É esse
catálogo que a Seção 7.1 usa como gabarito para corrigir a classificação
vinda do ORCID.

> **Nota de nomenclatura:** diferente da versão original de `scopus_2.ipynb`,
> aqui `id_scopus` **não** é preenchido com o `eid` (ID do artigo) retornado
> pela busca — em todo o resto do pipeline `id_scopus` sempre significou
> "Scopus Source ID" (ID do periódico, vindo da planilha de percentil na
> Seção 9). Preencher `id_scopus` com o `eid` aqui misturaria dois
> identificadores diferentes na mesma coluna; ele fica nulo até o
> cruzamento da Seção 9 preenchê-lo com o significado correto.

In [13]:
SCOPUS_API_KEY = __import__('os').getenv('SCOPUS_API_KEY')
SCOPUS_INST_TOKEN = __import__('os').getenv('SCOPUS_INST_TOKEN')

df_artigos_periodico_scopus = montar_df_vazio(COLUNAS_PERIODICO)
df_artigos_congresso_scopus = montar_df_vazio(COLUNAS_CONGRESSO)

if not SCOPUS_API_KEY:
    print("AVISO: SCOPUS_API_KEY não configurada no .env -- "
          "extração Scopus pulada (df_artigos_periodico_scopus/df_artigos_congresso_scopus ficam vazios).")
else:
    pybliometrics.init(keys=[SCOPUS_API_KEY], inst_tokens=[SCOPUS_INST_TOKEN] if SCOPUS_INST_TOKEN else None)
    print("Cliente Scopus (pybliometrics) inicializado com sucesso!")

Cliente Scopus (pybliometrics) inicializado com sucesso!


In [ ]:
def extrair_ano_scopus(cover_date):
    """Extrai o ano (int) de uma string de data tipo '2018-08-01'."""
    if not cover_date or pd.isna(cover_date):
        return pd.NA
    try:
        return int(str(cover_date)[:4])
    except (ValueError, TypeError):
        return pd.NA


# A classificação usa `subtype` (e não `aggregationType`): o agregador diz
# apenas em que *veículo* a publicação saiu ("Journal", "Conference
# Proceeding"), então uma carta ao editor publicada numa revista aparece como
# "Journal" igual a um artigo. O `subtype` diz o que a publicação *é*:
#   ar = Article        cp = Conference Paper   le = Letter
#   re = Review         ed = Editorial          no = Note
#   sh = Short Survey   ch = Chapter            er = Erratum   ...
# Só interessam ao pipeline artigos de periódico e trabalhos de congresso.
SUBTIPOS_PERIODICO_SCOPUS = {'ar'}
SUBTIPOS_CONGRESSO_SCOPUS = {'cp'}

# Colunas do catálogo Scopus — o registro de TUDO que a Scopus retornou,
# inclusive o que foi descartado. É ele que, na Seção 7.1, permite descobrir
# que um "journal_article" vindo do ORCID é na verdade uma carta.
COLUNAS_CATALOGO_SCOPUS = [
    'id_lattes', 'eid', 'titulo', 'doi', 'subtype', 'subtype_descricao',
    'aggregation_type', 'publicacao', 'aceito',
]

df_catalogo_scopus = montar_df_vazio(COLUNAS_CATALOGO_SCOPUS)

if SCOPUS_API_KEY:
    df_lista_pessoas_scopus = pd.read_csv(ARQUIVO_LISTA_PESSOAS, dtype=str) if Path(ARQUIVO_LISTA_PESSOAS).exists() else pd.DataFrame(columns=['id_lattes', 'scopus_author_id'])
    df_lista_pessoas_scopus = df_lista_pessoas_scopus[df_lista_pessoas_scopus['scopus_author_id'].notna() & (df_lista_pessoas_scopus['scopus_author_id'].str.strip() != '')]

    lista_artigos_periodico_scopus = []
    lista_artigos_congresso_scopus = []
    lista_catalogo_scopus = []

    print(f"Total de pessoas com scopus_author_id preenchido: {len(df_lista_pessoas_scopus)}")

    for _, pessoa in df_lista_pessoas_scopus.iterrows():
        id_lattes = str(pessoa['id_lattes']).strip()
        scopus_author_id = str(pessoa['scopus_author_id']).strip()

        print(f"Processando Scopus Author ID {scopus_author_id} (id_lattes={id_lattes})...")

        try:
            s = ScopusSearch(f'AU-ID({scopus_author_id})')
        except Exception as exc:
            print(f"  -> Falha ao buscar publicações de {scopus_author_id}: {exc}")
            continue

        publicacoes = s.results or []
        if not publicacoes:
            print("  -> Nenhuma publicação encontrada.")
            continue

        for pub in publicacoes:
            titulo = pub.title or pd.NA
            revista = pub.publicationName or pd.NA
            ano = extrair_ano_scopus(pub.coverDate)
            doi = pub.doi or pd.NA
            issn = pub.issn or pd.NA
            autores = pub.author_names or pd.NA
            subtipo = (getattr(pub, 'subtype', None) or '').strip().lower()
            subtipo_descricao = getattr(pub, 'subtypeDescription', None) or pd.NA
            tipo_agregacao = (pub.aggregationType or '').strip().lower()

            aceito = subtipo in SUBTIPOS_PERIODICO_SCOPUS or subtipo in SUBTIPOS_CONGRESSO_SCOPUS

            # O catálogo registra TODA publicação retornada, aceita ou não.
            lista_catalogo_scopus.append({
                'id_lattes': id_lattes, 'eid': getattr(pub, 'eid', None) or pd.NA,
                'titulo': titulo, 'doi': doi, 'subtype': subtipo or pd.NA,
                'subtype_descricao': subtipo_descricao, 'aggregation_type': tipo_agregacao or pd.NA,
                'publicacao': revista, 'aceito': aceito,
            })

            if subtipo in SUBTIPOS_CONGRESSO_SCOPUS:
                lista_artigos_congresso_scopus.append({
                    'id_lattes': id_lattes, 'titulo_artigo': titulo, 'ano': ano, 'doi': doi,
                    'autores': autores, 'titulo_evento_lattes': revista, 'paginas': pub.pageRange or pd.NA,
                    'sigla_evento_google': pd.NA, 'titulo_evento_google': pd.NA, 'estrato': pd.NA,
                    'tipo_match': pd.NA, 'coautoria_aluno': pd.NA, 'fonte': 'SCOPUS',
                })
            elif subtipo in SUBTIPOS_PERIODICO_SCOPUS:
                lista_artigos_periodico_scopus.append({
                    'id_lattes': id_lattes, 'titulo_artigo': titulo, 'titulo_revista_lattes': pd.NA,
                    'ano_pub': ano, 'doi': doi, 'autores': autores, 'match_adequado': pd.NA,
                    'coautoria_aluno': pd.NA, 'id_scopus': pd.NA, 'titulo_revista_scopus': revista,
                    'maior_percentil': pd.NA, 'codigo_area_maior_percentil': pd.NA,
                    'area_maior_percentil': pd.NA, 'issn': issn, 'computation_area': pd.NA, 'fonte': 'SCOPUS',
                })

        time.sleep(0.2)

    if lista_artigos_periodico_scopus:
        df_artigos_periodico_scopus = pd.DataFrame(lista_artigos_periodico_scopus)[COLUNAS_PERIODICO]
    if lista_artigos_congresso_scopus:
        df_artigos_congresso_scopus = pd.DataFrame(lista_artigos_congresso_scopus)[COLUNAS_CONGRESSO]
    if lista_catalogo_scopus:
        df_catalogo_scopus = pd.DataFrame(lista_catalogo_scopus)[COLUNAS_CATALOGO_SCOPUS]

    print(f"\nArtigos de periódico extraídos da Scopus: {len(df_artigos_periodico_scopus)}")
    print(f"Trabalhos de congresso extraídos da Scopus: {len(df_artigos_congresso_scopus)}")
    print(f"Catálogo Scopus (tudo que a API retornou): {len(df_catalogo_scopus)} publicações")

    if not df_catalogo_scopus.empty:
        print("\nDistribuição por subtipo (o que foi aceito e o que foi descartado):")
        display(
            df_catalogo_scopus.groupby(['subtype', 'subtype_descricao', 'aceito'], dropna=False)
            .size().rename('publicacoes').reset_index()
            .sort_values('publicacoes', ascending=False)
        )

## 7.1 Revisão da Classificação do ORCID pelo Catálogo Scopus

A API do ORCID descreve cada trabalho por um `type` autodeclarado por quem
cadastrou a publicação, e esse campo não separa bem os gêneros: cartas ao
editor, editoriais, notas e errata publicados em revista costumam chegar como
`journal_article` e entravam no pipeline como se fossem artigos. A Scopus, ao
contrário, traz o campo `subtype`/`subtypeDescription`, que diz o que a
publicação **é** (`ar` = Article, `cp` = Conference Paper, `le` = Letter,
`ed` = Editorial, `re` = Review, `no` = Note, `er` = Erratum, ...) —
diferente de `aggregationType`, que diz apenas em que veículo ela saiu e
portanto marca "Journal" tanto para um artigo quanto para uma carta.

Como praticamente tudo que está na Scopus também está no ORCID, o catálogo
Scopus (`df_catalogo_scopus`, montado na Seção 7 com **todas** as publicações
retornadas, inclusive as descartadas) serve de gabarito. Esta seção precisa
rodar depois das duas extrações — primeiro ORCID (Seção 6), depois Scopus
(Seção 7) — e para cada linha vinda do ORCID procura a publicação
correspondente no catálogo, primeiro por **DOI normalizado** e, se não
achar, por **título normalizado dentro do mesmo professor** (mesmas funções
de normalização usadas na deduplicação da Seção 8, ambas comparações exatas).

O desfecho depende do `subtype` encontrado:

| `subtype` no catálogo | Ação sobre a linha do ORCID |
| --- | --- |
| `ar` (Article) | fica em periódicos (movida para lá, se estava em congressos) |
| `cp` (Conference Paper) | fica em congressos (movida para lá, se estava em periódicos) |
| qualquer outro (`le`, `ed`, `no`, `er`, `re`, ...) | **descartada** — não é artigo nem trabalho de congresso |
| não encontrada no catálogo | mantida como veio (a Scopus não conhece a publicação, não há o que conferir) |

O relatório `df_revisao_orcid` lista linha a linha o que foi corrigido, com o
subtipo que motivou cada decisão, para conferência. A revisão só mexe nas
linhas de origem ORCID: Lattes e Scopus seguem intactos.

In [ ]:
# ---------------------------------------------------------------------------
# 7.1.1 Normalização de DOI e título
# ---------------------------------------------------------------------------
# Definidas aqui porque esta é a primeira seção que precisa delas; a
# deduplicação da Seção 8 usa exatamente as mesmas funções, para que "casar
# com o catálogo Scopus" e "ser a mesma publicação" sigam o mesmo critério.
def normalizar_doi(doi):
    """Reduz um DOI à sua forma canônica para comparação EXATA: minúsculo,
    sem espaços, sem prefixo textual ("doi:") nem de URL
    (https://doi.org/, http://dx.doi.org/) e sem pontuação solta na borda
    (várias fontes trazem o DOI seguido de ponto final). Retorna <NA> quando
    não sobra nada de útil — nesse caso a linha simplesmente não participa do
    casamento por DOI."""
    if pd.isna(doi):
        return pd.NA
    texto = str(doi).strip().lower()
    texto = re.sub(r'^doi\s*:\s*', '', texto)
    texto = re.sub(r'^https?://(dx\.)?doi\.org/', '', texto)
    texto = texto.strip(' .,;')
    if texto in ('', 'nan', 'none', '<na>'):
        return pd.NA
    return texto


def normalizar_titulo_dedup(titulo):
    """Reduz um título à sua forma canônica para comparação EXATA:
    maiúsculas, sem acentuação, sem pontuação e com espaços colapsados.
    Assim "Título: Algo Novo." (Lattes) e "TITULO ALGO NOVO" (Scopus) viram a
    mesma string. Retorna '' quando não há título — nesse caso a linha não
    participa do casamento por título."""
    if pd.isna(titulo):
        return ''
    texto = str(titulo).upper().strip()
    texto = unicodedata.normalize('NFKD', texto).encode('ascii', 'ignore').decode('ascii')
    texto = re.sub(r'[^A-Z0-9 ]+', ' ', texto)
    return re.sub(r'\s+', ' ', texto).strip()


# ---------------------------------------------------------------------------
# 7.1.2 Consulta ao catálogo Scopus
# ---------------------------------------------------------------------------
def indexar_catalogo_scopus(df_catalogo):
    """Monta dois índices de consulta a partir do catálogo: um por DOI
    normalizado (o DOI é único globalmente, então serve mesmo quando a
    publicação foi catalogada pelo coautor) e outro por título normalizado
    dentro do mesmo professor (título sozinho é ambíguo demais para casar
    entre professores diferentes)."""
    por_doi = {}
    por_titulo = {}
    for registro in df_catalogo.itertuples(index=False):
        subtipo = registro.subtype
        if pd.isna(subtipo) or not str(subtipo).strip():
            continue
        achado = (str(subtipo).strip().lower(), registro.subtype_descricao)

        doi = normalizar_doi(registro.doi)
        if not pd.isna(doi):
            por_doi.setdefault(doi, achado)

        titulo = normalizar_titulo_dedup(registro.titulo)
        if titulo:
            por_titulo.setdefault((str(registro.id_lattes), titulo), achado)
    return por_doi, por_titulo


def consultar_catalogo_scopus(linha, por_doi, por_titulo):
    """Procura no catálogo a publicação de uma linha do ORCID: primeiro pelo
    DOI, depois pelo título dentro do mesmo professor. Retorna
    (subtype, subtypeDescription) ou None se a Scopus não a conhece."""
    doi = normalizar_doi(linha.get('doi'))
    if not pd.isna(doi) and doi in por_doi:
        return por_doi[doi]

    titulo = normalizar_titulo_dedup(linha.get('titulo_artigo'))
    if titulo:
        return por_titulo.get((str(linha.get('id_lattes')), titulo))
    return None


# ---------------------------------------------------------------------------
# 7.1.3 Conversão entre os dois schemas (quando a linha muda de lado)
# ---------------------------------------------------------------------------
def converter_periodico_para_congresso(linha):
    """Reescreve uma linha do schema de periódico no schema de congresso —
    usada quando a Scopus revela que o que o ORCID chamou de artigo de
    periódico é na verdade um trabalho de congresso."""
    convertida = {coluna: pd.NA for coluna in COLUNAS_CONGRESSO}
    convertida.update({
        'id_lattes': linha['id_lattes'], 'titulo_artigo': linha['titulo_artigo'],
        'ano': linha['ano_pub'], 'doi': linha['doi'], 'autores': linha['autores'],
        'titulo_evento_lattes': linha['titulo_revista_lattes'], 'fonte': linha['fonte'],
    })
    return convertida


def converter_congresso_para_periodico(linha):
    """Caminho inverso de `converter_periodico_para_congresso`."""
    convertida = {coluna: pd.NA for coluna in COLUNAS_PERIODICO}
    convertida.update({
        'id_lattes': linha['id_lattes'], 'titulo_artigo': linha['titulo_artigo'],
        'ano_pub': linha['ano'], 'doi': linha['doi'], 'autores': linha['autores'],
        'titulo_revista_lattes': linha['titulo_evento_lattes'], 'fonte': linha['fonte'],
    })
    return convertida


# ---------------------------------------------------------------------------
# 7.1.4 A revisão propriamente dita
# ---------------------------------------------------------------------------
COLUNAS_REVISAO_ORCID = [
    'id_lattes', 'titulo_artigo', 'doi', 'classificacao_orcid',
    'subtype_scopus', 'subtype_descricao_scopus', 'acao',
]


def revisar_classificacao_orcid(df_periodico, df_congresso, df_catalogo):
    """Confere as linhas vindas do ORCID contra o catálogo Scopus e devolve
    (periódicos revisados, congressos revisados, relatório das mudanças).

    Linhas que a Scopus não conhece são mantidas exatamente como vieram — o
    catálogo só é usado para *corrigir* o que ele consegue afirmar, nunca
    para descartar por ausência de evidência."""
    if df_catalogo.empty:
        print("Catálogo Scopus vazio -- revisão pulada (nada com que conferir).")
        return df_periodico, df_congresso, montar_df_vazio(COLUNAS_REVISAO_ORCID)

    por_doi, por_titulo = indexar_catalogo_scopus(df_catalogo)

    periodicos_revisados = []
    congressos_revisados = []
    ocorrencias = []

    for origem, df_origem in [('PERIODICO', df_periodico), ('CONGRESSO', df_congresso)]:
        for linha in df_origem.to_dict('records'):
            achado = consultar_catalogo_scopus(linha, por_doi, por_titulo)

            if achado is None:
                destino = origem  # a Scopus não conhece: mantém como veio
                subtipo, subtipo_descricao = pd.NA, pd.NA
            else:
                subtipo, subtipo_descricao = achado
                if subtipo in SUBTIPOS_PERIODICO_SCOPUS:
                    destino = 'PERIODICO'
                elif subtipo in SUBTIPOS_CONGRESSO_SCOPUS:
                    destino = 'CONGRESSO'
                else:
                    destino = 'DESCARTADO'

            if destino == 'PERIODICO':
                periodicos_revisados.append(
                    linha if origem == 'PERIODICO' else converter_congresso_para_periodico(linha)
                )
            elif destino == 'CONGRESSO':
                congressos_revisados.append(
                    linha if origem == 'CONGRESSO' else converter_periodico_para_congresso(linha)
                )

            if destino != origem:
                ocorrencias.append({
                    'id_lattes': linha['id_lattes'], 'titulo_artigo': linha['titulo_artigo'],
                    'doi': linha['doi'], 'classificacao_orcid': origem,
                    'subtype_scopus': subtipo, 'subtype_descricao_scopus': subtipo_descricao,
                    'acao': 'DESCARTADO' if destino == 'DESCARTADO' else f'MOVIDO PARA {destino}',
                })

    df_periodico_revisado = (
        pd.DataFrame(periodicos_revisados)[COLUNAS_PERIODICO] if periodicos_revisados
        else montar_df_vazio(COLUNAS_PERIODICO)
    )
    df_congresso_revisado = (
        pd.DataFrame(congressos_revisados)[COLUNAS_CONGRESSO] if congressos_revisados
        else montar_df_vazio(COLUNAS_CONGRESSO)
    )
    df_relatorio = (
        pd.DataFrame(ocorrencias)[COLUNAS_REVISAO_ORCID] if ocorrencias
        else montar_df_vazio(COLUNAS_REVISAO_ORCID)
    )
    return df_periodico_revisado.reset_index(drop=True), df_congresso_revisado.reset_index(drop=True), df_relatorio


print("Conferindo a classificação do ORCID contra o catálogo Scopus...")
print(f"  Antes -- ORCID periódicos: {len(df_artigos_periodico_orcid)} | "
      f"ORCID congressos: {len(df_artigos_congresso_orcid)}")

df_artigos_periodico_orcid, df_artigos_congresso_orcid, df_revisao_orcid = revisar_classificacao_orcid(
    df_artigos_periodico_orcid, df_artigos_congresso_orcid, df_catalogo_scopus,
)

print(f"  Depois -- ORCID periódicos: {len(df_artigos_periodico_orcid)} | "
      f"ORCID congressos: {len(df_artigos_congresso_orcid)}")

if df_revisao_orcid.empty:
    print("Nenhuma correção necessária: tudo que a Scopus conhece já estava classificado corretamente.")
else:
    print(f"\n{len(df_revisao_orcid)} linha(s) do ORCID corrigida(s):")
    display(df_revisao_orcid['acao'].value_counts())
    print("\nDescartes por subtipo Scopus:")
    display(
        df_revisao_orcid[df_revisao_orcid['acao'] == 'DESCARTADO']
        .groupby(['subtype_scopus', 'subtype_descricao_scopus'], dropna=False)
        .size().rename('linhas').reset_index().sort_values('linhas', ascending=False)
    )
    display(df_revisao_orcid.head(20))

## 8. Unificação das Três Fontes com Deduplicação

As três fontes produzem exatamente o mesmo schema
(`COLUNAS_PERIODICO`/`COLUNAS_CONGRESSO`) e são simplesmente concatenadas.
O resultado bruto (`df_periodicos_bruto`/`df_congressos_bruto`) **preserva
uma linha por publicação por fonte** — é essa granularidade que a Seção 12
propaga de volta para as 6 tabelas por fonte.

### A regra que orienta toda a seção

A deduplicação é feita **dentro da lista de cada professor, isoladamente**.
O agrupamento é por `id_lattes` e a chave gerada leva o `id_lattes` como
prefixo, de modo que duas linhas de professores diferentes **nunca** podem
cair na mesma chave. Consequência prática: se dois professores do quadro
coassinaram o mesmo artigo, os dois continuam com aquele artigo na sua
própria lista — a pergunta "quais artigos este professor tem?" continua
respondível. O que se elimina é apenas a repetição **dentro** da lista de um
mesmo professor.

### Como duas linhas do mesmo professor viram a mesma publicação

Só por comparação **exata**, sobre valores previamente tratados — não há
nenhum limiar de similaridade em nenhuma etapa:

- mesmo **DOI normalizado** (`normalizar_doi`: minúsculo, sem `doi:`, sem
  `https://doi.org/`, sem pontuação na borda); ou
- mesmo **título normalizado** (`normalizar_titulo_dedup`: maiúsculas, sem
  acentuação, sem pontuação, espaços colapsados).

As duas funções de normalização são as definidas na Seção 7.1 — o mesmo
critério de igualdade que casa uma linha do ORCID com o catálogo Scopus.

Os dois critérios são combinados por componentes conexos porque cada fonte
preenche campos diferentes: se o Lattes trouxe o título sem DOI e o Scopus
trouxe o mesmo título com DOI, é o título que liga as duas linhas; se uma
terceira linha do ORCID só tem o DOI, é o DOI que a liga à do Scopus. As
três acabam na mesma publicação. A chave final é `id_lattes|DOI:<doi>`
quando o grupo tem DOI, e `id_lattes|TIT:<titulo>` quando não tem.

### As duas reduções

1. `deduplicar_bruto_por_fonte` deixa uma linha por (publicação, fonte) na
   base bruta — garante que as 6 tabelas por fonte, usadas nos relatórios,
   também não tenham artigos repetidos.
2. `unificar_com_dedup` reduz à base final: uma linha por publicação de cada
   professor, com cada coluna preenchida pelo primeiro valor não nulo na
   ordem LATTES > SCOPUS > ORCID (Lattes é curado manualmente pelo professor;
   Scopus tem metadados mais limpos que o resumo público do ORCID). A coluna
   `fontes` registra todas as bases em que aquela publicação foi encontrada.

### Segunda passada

`auditar_duplicatas` percorre as listas de novo, sem alterar nada, checando
os cinco invariantes: nenhum professor com DOI repetido, nenhum professor com
título repetido, nenhuma chave repetida na base unificada, nenhuma linha
repetida dentro da mesma fonte na base bruta e nenhuma chave compartilhada
por professores diferentes (este último confirma que a coautoria entre
professores do quadro não fundiu ninguém).

In [ ]:
# `normalizar_doi` e `normalizar_titulo_dedup` são definidas na Seção 7.1, que
# roda antes desta e precisa das mesmas normalizações para casar as linhas do
# ORCID com o catálogo Scopus. A deduplicação reusa exatamente aquelas funções,
# para que os dois cruzamentos sigam o mesmo critério de igualdade.


# ---------------------------------------------------------------------------
# 8.2 Chave de deduplicação — SEMPRE dentro de um único professor
# ---------------------------------------------------------------------------
def calcular_chave_dedup(df):
    """Atribui a cada linha a chave da publicação a que ela pertence.

    Regra fundamental: a deduplicação acontece **dentro da lista de cada
    professor**, nunca entre professores. As linhas são agrupadas por
    `id_lattes` e o casamento (por DOI ou por título) só é procurado dentro
    do grupo; a chave gerada carrega o `id_lattes` como prefixo, então duas
    linhas de professores diferentes nunca podem cair na mesma chave. Se dois
    professores do quadro coassinaram o mesmo artigo, cada um mantém a sua
    própria linha daquele artigo — o que se elimina é apenas a repetição
    dentro da lista de um mesmo professor.

    Dentro do grupo de um professor, duas linhas são a mesma publicação se:
      - têm o mesmo DOI normalizado (comparação exata), OU
      - têm o mesmo título normalizado (comparação exata).
    Nenhum limiar de similaridade é usado em nenhum dos dois critérios.

    Os dois critérios são combinados por união (componentes conexos) porque
    fontes diferentes preenchem campos diferentes: se o Lattes trouxe o
    título sem DOI e o Scopus trouxe o mesmo título com DOI, é o título que
    liga as duas linhas; e se uma terceira linha do ORCID só tem o DOI, é o
    DOI que a liga à do Scopus. As três acabam no mesmo grupo.

    A chave final de cada grupo é `id_lattes|DOI:<doi>` quando alguma linha do
    grupo tem DOI (é o identificador mais forte) e `id_lattes|TIT:<titulo>`
    caso contrário. Linhas sem DOI e sem título não casam com nada e recebem
    uma chave própria (`id_lattes|LINHA:<n>`), para não serem fundidas entre si.
    """
    total = len(df)
    if total == 0:
        return pd.Series([], dtype='object', index=df.index)

    dois = df['doi'].map(normalizar_doi).tolist()
    titulos = df['titulo_artigo'].map(normalizar_titulo_dedup).tolist()
    professores = df['id_lattes'].astype(str).tolist()

    # Agrupa as posições das linhas por professor: cada grupo é deduplicado
    # isoladamente, sem enxergar as linhas dos demais professores.
    posicoes_por_professor = {}
    for posicao in range(total):
        posicoes_por_professor.setdefault(professores[posicao], []).append(posicao)

    chaves = [None] * total

    for professor, posicoes in posicoes_por_professor.items():
        pai = {posicao: posicao for posicao in posicoes}

        def encontrar(x):
            while pai[x] != x:
                pai[x] = pai[pai[x]]
                x = pai[x]
            return x

        def unir(a, b):
            raiz_a, raiz_b = encontrar(a), encontrar(b)
            if raiz_a != raiz_b:
                pai[raiz_b] = raiz_a

        # Uma passada só: cada linha se une à primeira linha do professor que
        # tenha o mesmo DOI e à primeira que tenha o mesmo título.
        primeira_com_doi = {}
        primeira_com_titulo = {}
        for posicao in posicoes:
            doi = dois[posicao]
            if not pd.isna(doi):
                unir(primeira_com_doi.setdefault(doi, posicao), posicao)
            titulo = titulos[posicao]
            if titulo:
                unir(primeira_com_titulo.setdefault(titulo, posicao), posicao)

        # DOI representante de cada grupo (o menor, para ser determinístico
        # caso um grupo tenha sido ligado por título e traga dois DOIs).
        doi_do_grupo = {}
        for posicao in posicoes:
            doi = dois[posicao]
            if pd.isna(doi):
                continue
            raiz = encontrar(posicao)
            if raiz not in doi_do_grupo or doi < doi_do_grupo[raiz]:
                doi_do_grupo[raiz] = doi

        for posicao in posicoes:
            raiz = encontrar(posicao)
            if raiz in doi_do_grupo:
                identificador = f'DOI:{doi_do_grupo[raiz]}'
            elif titulos[raiz]:
                identificador = f'TIT:{titulos[raiz]}'
            else:
                identificador = f'LINHA:{raiz}'
            chaves[posicao] = f'{professor}|{identificador}'

    return pd.Series(chaves, index=df.index, dtype='object')


# ---------------------------------------------------------------------------
# 8.3 Redução: uma linha por publicação por fonte, e uma linha por publicação
# ---------------------------------------------------------------------------
# Lattes é a fonte curada manualmente pelo próprio professor; Scopus tem
# metadados bibliográficos mais limpos que o resumo público do ORCID.
ORDEM_PRIORIDADE_FONTE = {'LATTES': 0, 'SCOPUS': 1, 'ORCID': 2}


def deduplicar_bruto_por_fonte(df, chave='chave_dedup'):
    """Garante uma única linha por (publicação, fonte) na base bruta — que é
    a base propagada de volta na Seção 12 para as 6 tabelas por fonte. Sem
    isso, uma repetição vinda da própria extração (o mesmo artigo listado
    duas vezes no currículo Lattes, por exemplo) sobreviveria até as tabelas
    usadas nos relatórios. Mantém a primeira ocorrência, respeitando a ordem
    de prioridade das fontes."""
    if df.empty:
        return df
    df_ordenado = df.copy()
    df_ordenado['_prioridade'] = df_ordenado['fonte'].map(ORDEM_PRIORIDADE_FONTE).fillna(9)
    df_ordenado = df_ordenado.sort_values('_prioridade', kind='stable')
    return (
        df_ordenado.drop_duplicates(subset=['fonte', chave], keep='first')
        .drop(columns=['_prioridade'])
        .reset_index(drop=True)
    )


def unificar_com_dedup(df, chave='chave_dedup'):
    """Reduz a base bruta (uma linha por publicação por fonte) à base final
    (uma linha por publicação de cada professor). Cada coluna recebe o
    primeiro valor não nulo encontrado no grupo, na ordem de prioridade das
    fontes, e a coluna `fontes` registra todas as bases em que aquela
    publicação foi encontrada — é ela que permite, depois, apontar em quais
    bases o artigo está faltando."""
    if df.empty:
        return pd.DataFrame(columns=[c for c in df.columns if c != 'fonte'] + ['fontes'])

    df_ordenado = df.copy()
    df_ordenado['_prioridade'] = df_ordenado['fonte'].map(ORDEM_PRIORIDADE_FONTE).fillna(9)
    df_ordenado = df_ordenado.sort_values('_prioridade', kind='stable')

    colunas_dado = [c for c in df.columns if c not in (chave, 'fonte', '_prioridade')]
    linhas_unicas = df_ordenado.drop_duplicates(subset=[chave], keep='first')[[chave] + colunas_dado].copy()

    for coluna in colunas_dado:
        if not linhas_unicas[coluna].isna().any():
            continue
        preenchimento = (
            df_ordenado.dropna(subset=[coluna])
            .drop_duplicates(subset=[chave], keep='first')[[chave, coluna]]
            .rename(columns={coluna: '_preenchimento'})
        )
        linhas_unicas = linhas_unicas.merge(preenchimento, on=chave, how='left')
        linhas_unicas[coluna] = linhas_unicas[coluna].fillna(linhas_unicas['_preenchimento'])
        linhas_unicas = linhas_unicas.drop(columns=['_preenchimento'])

    fontes_por_publicacao = (
        df.groupby(chave)['fonte']
        .apply(lambda serie: ','.join(sorted(set(serie.dropna()))))
        .rename('fontes')
        .reset_index()
    )
    return linhas_unicas.merge(fontes_por_publicacao, on=chave, how='left').reset_index(drop=True)


# ---------------------------------------------------------------------------
# 8.4 Segunda passada: percorre as listas de novo procurando deslizes
# ---------------------------------------------------------------------------
def auditar_duplicatas(df_unificado, df_bruto, rotulo):
    """Confere (sem alterar nada) se o resultado bate com os requisitos:

    1. nenhum professor tem o mesmo DOI repetido na base unificada;
    2. nenhum professor tem o mesmo título repetido na base unificada;
    3. nenhuma chave se repete na base unificada;
    4. nenhuma linha se repete dentro da mesma fonte na base bruta (é ela que
       vira as 6 tabelas por fonte);
    5. nenhuma chave é compartilhada por dois professores diferentes — ou
       seja, a coautoria entre professores do quadro não fundiu ninguém.
    """
    doi_norm = df_unificado['doi'].map(normalizar_doi)
    titulo_norm = df_unificado['titulo_artigo'].map(normalizar_titulo_dedup)
    professor = df_unificado['id_lattes'].astype(str)

    dup_doi = pd.concat([professor, doi_norm], axis=1).duplicated(keep=False) & doi_norm.notna()
    dup_titulo = (professor + '|' + titulo_norm).duplicated(keep=False) & (titulo_norm != '')
    dup_chave = df_unificado['chave_dedup'].duplicated(keep=False)
    dup_fonte = df_bruto.duplicated(subset=['fonte', 'chave_dedup'], keep=False)
    chaves_multiprofessor = (
        df_unificado.groupby('chave_dedup')['id_lattes'].nunique().gt(1).sum()
    )

    total_problemas = (
        int(dup_doi.sum()) + int(dup_titulo.sum()) + int(dup_chave.sum())
        + int(dup_fonte.sum()) + int(chaves_multiprofessor)
    )
    if total_problemas == 0:
        print(f"  OK ({rotulo}): nenhum professor tem artigo repetido, nem na base "
              f"unificada nem nas bases por fonte; nenhuma chave cruzou professores.")
        return

    if dup_doi.any():
        print(f"  AVISO ({rotulo}): {int(dup_doi.sum())} linha(s) com o mesmo DOI para o mesmo professor:")
        display(df_unificado[dup_doi][['id_lattes', 'titulo_artigo', 'doi', 'chave_dedup', 'fontes']]
                .sort_values(['id_lattes', 'doi']))
    if dup_titulo.any():
        print(f"  AVISO ({rotulo}): {int(dup_titulo.sum())} linha(s) com o mesmo título para o mesmo professor:")
        display(df_unificado[dup_titulo][['id_lattes', 'titulo_artigo', 'doi', 'chave_dedup', 'fontes']]
                .sort_values(['id_lattes', 'titulo_artigo']))
    if dup_chave.any():
        print(f"  AVISO ({rotulo}): {int(dup_chave.sum())} linha(s) com chave_dedup repetida na base unificada.")
    if dup_fonte.any():
        print(f"  AVISO ({rotulo}): {int(dup_fonte.sum())} linha(s) repetida(s) dentro da mesma fonte (base bruta):")
        display(df_bruto[dup_fonte][['id_lattes', 'fonte', 'titulo_artigo', 'doi', 'chave_dedup']]
                .sort_values(['id_lattes', 'fonte', 'titulo_artigo']))
    if chaves_multiprofessor:
        print(f"  AVISO ({rotulo}): {int(chaves_multiprofessor)} chave(s) compartilhada(s) por professores diferentes.")


# ---------------------------------------------------------------------------
# 8.5 Execução
# ---------------------------------------------------------------------------
print("Concatenando as três fontes (uma linha por publicação por fonte)...")
df_periodicos_bruto = pd.concat(
    [df_artigos_periodico_lattes, df_artigos_periodico_orcid, df_artigos_periodico_scopus],
    ignore_index=True,
)
df_congressos_bruto = pd.concat(
    [df_artigos_congresso_lattes, df_artigos_congresso_orcid, df_artigos_congresso_scopus],
    ignore_index=True,
)

print("Calculando a chave de deduplicação (por professor, via DOI ou título)...")
df_periodicos_bruto['chave_dedup'] = calcular_chave_dedup(df_periodicos_bruto)
df_congressos_bruto['chave_dedup'] = calcular_chave_dedup(df_congressos_bruto)

print("Removendo repetições dentro de uma mesma fonte...")
df_periodicos_bruto = deduplicar_bruto_por_fonte(df_periodicos_bruto)
df_congressos_bruto = deduplicar_bruto_por_fonte(df_congressos_bruto)

print("Unificando as fontes (uma linha por publicação de cada professor)...")
df_periodicos_unificado = unificar_com_dedup(df_periodicos_bruto)
df_congressos_unificado = unificar_com_dedup(df_congressos_bruto)

print("\nSegunda passada — conferindo se sobrou alguma duplicata:")
auditar_duplicatas(df_periodicos_unificado, df_periodicos_bruto, 'periódicos')
auditar_duplicatas(df_congressos_unificado, df_congressos_bruto, 'congressos')

print(f"\nPeriódicos -- bruto (por fonte): {len(df_periodicos_bruto)} | unificado: {len(df_periodicos_unificado)}")
print(f"Congressos -- bruto (por fonte): {len(df_congressos_bruto)} | unificado: {len(df_congressos_unificado)}")
print("\nCobertura por combinação de fontes (periódicos):")
display(df_periodicos_unificado['fontes'].value_counts())
display(df_periodicos_unificado[['id_lattes', 'titulo_artigo', 'doi', 'fontes']].head())

## 9. Cruzamento de Periódicos com a Base de Percentil Scopus

Mesma lógica em camadas da versão anterior deste notebook (match exato por
nome, match exato por ISSN, depois busca bidirecional por substring apenas
no que sobrou), agora aplicada sobre `df_periodicos_unificado` em vez de
`df_bib_artigos` isolado. A única mudança estrutural é a chave de nome
usada para o match: `COALESCE(titulo_revista_lattes, titulo_revista_scopus)`
— assim um artigo de origem Scopus (que já chega com `titulo_revista_scopus`
preenchido, mas sem `titulo_revista_lattes`) participa do mesmo cruzamento
que um artigo de origem Lattes ou ORCID.

Cada linha é identificada pela `chave_dedup` (calculada na Seção 8) em vez
de `titulo + id_lattes` como na versão anterior — mesma ideia, mas usando a
chave que já garante uma linha por publicação real.

> **Nota de fidelidade ao comportamento original:** quando tanto o `issn` da
> base unificada quanto `E-ISSN`/`Print ISSN` (Scopus) são nulos, o
> `pd.merge` do pandas trata os dois nulos como iguais e gera um match "por
> ISSN" mesmo sem nenhum ISSN de fato existir nos dois lados. Esse
> comportamento já existia no notebook original e é preservado aqui.


In [16]:
def formatar_issn(issn):
    """Normaliza um ISSN para 8 dígitos sem hífen, retornando nulo se o valor for vazio/inválido."""
    issn_str = str(issn).replace('-', '').strip()
    if issn_str.lower() in ['nan', 'none', '', 'nat']:
        return pd.NA
    return issn_str.zfill(8)


COLUNAS_PERIODICO_UNIFICADO = [c for c in COLUNAS_PERIODICO if c != 'fonte'] + ['fontes', 'chave_dedup']

print("Carregando e preparando a base completa da Scopus...")
df_scopus_raw = pd.read_excel(ARQUIVO_PERCENTIL_SCOPUS)
df_scopus_raw['Title'] = df_scopus_raw['Title'].astype(str).str.upper().str.strip()
df_scopus_raw['E-ISSN'] = df_scopus_raw['E-ISSN'].apply(formatar_issn)
df_scopus_raw['Print ISSN'] = df_scopus_raw['Print ISSN'].apply(formatar_issn)

mask_comput = df_scopus_raw['Scopus Sub-Subject Area'].str.contains('Comput', case=False, na=False)
titulos_computacao = set(df_scopus_raw[mask_comput]['Title'].unique())
issns_computacao = set(df_scopus_raw[mask_comput]['E-ISSN'].dropna().unique()).union(
                    set(df_scopus_raw[mask_comput]['Print ISSN'].dropna().unique()))

# Um mesmo periódico pode aparecer várias vezes na planilha (uma linha por
# subárea ASJC). Mantemos apenas o percentil mais alto de cada título.
df_scopus_unicos = df_scopus_raw.sort_values(by='Percentile', ascending=False)
df_scopus_unicos = df_scopus_unicos.drop_duplicates(subset=['Title'], keep='first').copy()

colunas_scopus = [
    'Scopus Source ID', 'Title', 'Percentile',
    'Scopus ASJC Code (Sub-subject Area)', 'Scopus Sub-Subject Area', 'E-ISSN', 'Print ISSN'
]
df_scopus_filtro = df_scopus_unicos[colunas_scopus]

print("Preparando a base unificada de periódicos (chave de cruzamento)...")
df_base_periodicos = df_periodicos_unificado.copy()
df_base_periodicos['revista'] = (
    df_base_periodicos['titulo_revista_lattes']
    .where(df_base_periodicos['titulo_revista_lattes'].notna(), df_base_periodicos['titulo_revista_scopus'])
    .astype(str).str.upper().str.strip()
)
df_base_periodicos['issn'] = df_base_periodicos['issn'].apply(formatar_issn)

print("Preparação concluída.")

Carregando e preparando a base completa da Scopus...
Preparando a base unificada de periódicos (chave de cruzamento)...
Preparação concluída.


### 9.1 Camadas 1 e 2 — Match exato (nome da revista e ISSN)

In [17]:
print("Realizando o cruzamento exato (ISSN e Nome Exato)...")

df_match_nome = pd.merge(df_base_periodicos, df_scopus_filtro, left_on='revista', right_on='Title', how='inner')
df_match_e_issn = pd.merge(df_base_periodicos, df_scopus_filtro, left_on='issn', right_on='E-ISSN', how='inner')
df_match_print_issn = pd.merge(df_base_periodicos, df_scopus_filtro, left_on='issn', right_on='Print ISSN', how='inner')

df_sucessos = pd.concat([df_match_nome, df_match_e_issn, df_match_print_issn], ignore_index=True)
df_sucessos = df_sucessos.drop_duplicates(subset=['chave_dedup']).copy()

df_sucessos['Computation Area'] = (
    df_sucessos['Title'].isin(titulos_computacao) |
    df_sucessos['E-ISSN'].isin(issns_computacao) |
    df_sucessos['Print ISSN'].isin(issns_computacao)
)

print(f"Matches exatos encontrados: {len(df_sucessos)}")

Realizando o cruzamento exato (ISSN e Nome Exato)...
Matches exatos encontrados: 1928


### 9.2 Camada 3 — Match bidirecional por substring (apenas no restante)

In [18]:
print("Busca bidirecional (nome contido) para os artigos restantes...")

chaves_com_match_exato = set(df_sucessos['chave_dedup'])
df_restante = df_base_periodicos[~df_base_periodicos['chave_dedup'].isin(chaves_com_match_exato)].copy()

# Ordena a lista de periódicos Scopus do nome mais longo para o mais curto,
# para evitar que um nome curto "roube" o match de um nome mais específico.
df_scopus_filtro_sorted = df_scopus_filtro.copy()
df_scopus_filtro_sorted['tamanho_titulo'] = df_scopus_filtro_sorted['Title'].str.len()
df_scopus_filtro_sorted = df_scopus_filtro_sorted.sort_values(by='tamanho_titulo', ascending=False)
lista_scopus = df_scopus_filtro_sorted.to_dict('records')


def busca_bidirecional_revista(revista_lattes):
    """Procura, na lista Scopus, um título que contenha (ou esteja contido em) o nome da revista."""
    if pd.isna(revista_lattes) or revista_lattes == 'NAN' or revista_lattes == '':
        return None

    for scopus in lista_scopus:
        titulo_scopus = scopus['Title']
        if pd.notna(titulo_scopus) and titulo_scopus != 'NAN' and titulo_scopus != "":
            if (titulo_scopus in revista_lattes) or (revista_lattes in titulo_scopus):
                return scopus
    return None


resultados_parciais = df_restante['revista'].apply(busca_bidirecional_revista)

mask_encontrados = resultados_parciais.notna()
df_match_parcial = df_restante[mask_encontrados].copy()

if not df_match_parcial.empty:
    dicts_encontrados = resultados_parciais[mask_encontrados]
    for col in colunas_scopus:
        df_match_parcial[col] = [d[col] for d in dicts_encontrados]

    df_match_parcial['Computation Area'] = (
        df_match_parcial['Title'].isin(titulos_computacao) |
        df_match_parcial['E-ISSN'].isin(issns_computacao) |
        df_match_parcial['Print ISSN'].isin(issns_computacao)
    )

    df_sucessos = pd.concat([df_sucessos, df_match_parcial], ignore_index=True)
    df_sucessos = df_sucessos.drop_duplicates(subset=['chave_dedup']).copy()

print(f"Total de sucessos após a busca bidirecional: {len(df_sucessos)}")

Busca bidirecional (nome contido) para os artigos restantes...
Total de sucessos após a busca bidirecional: 2244


### 9.3 Falhas de match e consolidação final

In [19]:
print("Isolando e tratando as falhas definitivas...")

df_falhas = df_restante[~mask_encontrados].copy()
df_falhas['Scopus Source ID'] = pd.NA
df_falhas['Title'] = pd.NA
df_falhas['Percentile'] = 0
df_falhas['Scopus ASJC Code (Sub-subject Area)'] = pd.NA
df_falhas['Scopus Sub-Subject Area'] = pd.NA
df_falhas['E-ISSN'] = pd.NA
df_falhas['Print ISSN'] = pd.NA
df_falhas['Computation Area'] = False

print("Consolidando o resultado do cruzamento...")
df_periodicos_tratado = pd.concat([df_sucessos, df_falhas], ignore_index=True)
df_periodicos_tratado['Percentile'] = df_periodicos_tratado['Percentile'].astype(int)
df_periodicos_tratado['Computation Area'] = df_periodicos_tratado['Computation Area'].astype(bool)

# 'match_adequado' indica se o artigo encontrou correspondência válida na Scopus
df_periodicos_tratado['match_adequado'] = df_periodicos_tratado['Scopus Source ID'].notna()

# Sobrescreve os campos de cruzamento com o resultado, mantendo os demais
# campos (id_lattes, titulo_artigo, ano_pub, doi, autores, fontes, chave_dedup) intactos.
df_periodicos_tratado['id_scopus'] = df_periodicos_tratado['Scopus Source ID']
df_periodicos_tratado['titulo_revista_scopus'] = df_periodicos_tratado['Title']
df_periodicos_tratado['maior_percentil'] = df_periodicos_tratado['Percentile']
df_periodicos_tratado['codigo_area_maior_percentil'] = df_periodicos_tratado['Scopus ASJC Code (Sub-subject Area)']
df_periodicos_tratado['area_maior_percentil'] = df_periodicos_tratado['Scopus Sub-Subject Area']
df_periodicos_tratado['issn'] = df_periodicos_tratado['E-ISSN']
df_periodicos_tratado['computation_area'] = df_periodicos_tratado['Computation Area']

df_periodicos_unificado = df_periodicos_tratado[COLUNAS_PERIODICO_UNIFICADO].copy()

print(f"Tabela unificada de periódicos tratada. Total de linhas: {len(df_periodicos_unificado)}")
display(df_periodicos_unificado.sample(min(10, len(df_periodicos_unificado))))

Isolando e tratando as falhas definitivas...
Consolidando o resultado do cruzamento...
Tabela unificada de periódicos tratada. Total de linhas: 2311


,id_lattes,titulo_artigo,titulo_revista_lattes,ano_pub,doi,autores,match_adequado,coautoria_aluno,id_scopus,titulo_revista_scopus,maior_percentil,codigo_area_maior_percentil,area_maior_percentil,issn,computation_area,fontes,chave_dedup
1219,2291334095539768,Optimizing surplus hydropower sales under liqu...,NaN,2027,10.1016/j.epsr.2026.113578,"Clímaco, Francisco Glaubos Nunes;Sabóia, Ana L...",True,NaN,16044,ELECTRIC POWER SYSTEMS RESEARCH,84,2208,Electrical and Electronic Engineering,NaN,False,"ORCID,SCOPUS",2291334095539768|DOI:10.1016/j.epsr.2026.113578
1797,4436183480921146,Lagrangean decomposition in integer linear pro...,INFOR. INFORMATION SYSTEMS AND OPERATIONAL RES...,1992,NaN,"MACULAN FILHO, N.; REINOSO, H.",True,NaN,14363,INFOR,35,1803,Management Science and Operations Research,19160615,False,LATTES,4436183480921146|TIT:LAGRANGEAN DECOMPOSITION ...
144,2002515486942024,On neighborhood-Helly graphs,DISCRETE APPLIED MATHEMATICS,2017,http://dx.doi.org/10.1016/j.dam.2016.04.029,"GROSHAUS, Marina ; LIN, MIN CHIH ; Szwarcfiter...",True,NaN,25890,DISCRETE APPLIED MATHEMATICS,73,2607,Discrete Mathematics and Combinatorics,NaN,False,"LATTES,SCOPUS",2002515486942024|DOI:10.1016/j.dam.2016.04.029
1434,3957046121364560,"On the diameter of the Cayley Graph Hℓ,p",NaN,2015,10.21711/231766362015/rmc443,NaN,True,NaN,29346,SOUTH ATLANTIC QUARTERLY,99,1208,Literature and Literary Theory,NaN,False,ORCID,3957046121364560|DOI:10.21711/231766362015/rmc443
618,4602221579308599,Leveraging the partition selection bias to ach...,JOURNAL OF PROTEOMICS,2021,http://dx.doi.org/10.1016/j.jprot.2021.104282,"SILVA, A. R. F. ; LIMA, D. B. ; KURT, L. U. ; ...",True,NaN,11700154304,JOURNAL OF PROTEOMICS,79,1304,Biophysics,18767737,False,LATTES,4602221579308599|DOI:10.1016/j.jprot.2021.104282
1651,2291334095539768,Planning and Scheduling a Fleet of Rigs Using ...,COMPUTERS & INDUSTRIAL ENGINEERING,2012,http://dx.doi.org/10.1016/j.cie.2012.08.001,"Bassi, H. V. ; BAHIENSE, L. ; Ferreira Filho, ...",True,NaN,18164,COMPUTERS AND INDUSTRIAL ENGINEERING,96,2200,Engineering (all),NaN,True,"LATTES,ORCID,SCOPUS",2291334095539768|DOI:10.1016/j.cie.2012.08.001
1539,6243465206463403,Efficient allocation of resources in multiple ...,JOURNAL OF PARALLEL AND DISTRIBUTED COMPUTING ...,2014,http://dx.doi.org/10.1016/j.jpdc.2013.09.012,"LI, WEI ; Delicato, Flávia C. ; PIRES, PAULO F...",True,NaN,25621,JOURNAL OF PARALLEL AND DISTRIBUTED COMPUTING,91,2614,Theoretical Computer Science,10960848,True,LATTES,6243465206463403|DOI:10.1016/j.jpdc.2013.09.012
1872,5171924915397166,MylynSDP - Process - aware artifact filtering ...,JOURNAL OF THE BRAZILIAN COMPUTER SOCIETY (IMP...,2020,http://dx.doi.org/10.1186/s13173-020-00100-8,"PORTUGAL, IVENS ; OLIVEIRA, TOACY ; ALENCAR, P...",True,NaN,145262,JOURNAL OF THE BRAZILIAN COMPUTER SOCIETY,38,1700,Computer Science (all),16784804,True,LATTES,5171924915397166|DOI:10.1186/s13173-020-00100-8
1173,2002515486942024,Characterizing intersection graphs of substars...,NaN,2006,NaN,"Cerioli, Márcia R.;Szwarcfiter, Jayme L.",True,NaN,25215,ARS COMBINATORIA,4,2600,Mathematics (all),28175204,False,SCOPUS,2002515486942024|TIT:CHARACTERIZING INTERSECTI...
2310,5727472788265998,Estudo Introdutório do Protocolo Quântico BB84...,REIC. REVISTA ELETRÔNICA DE INICIAÇÃO CIENTÍFICA,2004,NaN,"MARQUEZINO, F.L.; HELAYËL-NETO, J.A.",False,NaN,<NA>,<NA>,0,<NA>,<NA>,<NA>,False,LATTES,5727472788265998|TIT:ESTUDO INTRODUTORIO DO PR...


## 10. Cruzamento de Trabalhos de Congresso com a Base de Eventos Classificados

Mesma lógica de match fuzzy (`rapidfuzz`) da versão anterior, agora sobre
`df_congressos_unificado`. Como o schema final já usa `titulo_evento_lattes`
como nome do campo de evento (preenchido a partir do nome do periódico/evento
retornado por qualquer uma das três fontes — não só o Lattes), o cruzamento
funciona igual para trabalhos vindos de ORCID/Scopus, sem precisar de nenhuma
adaptação de nome de coluna.


### 10.1 Carga e padronização da base de eventos classificados (Google/CAPES)

In [20]:
print("Carregando a base de eventos classificados...")
df_google_raw = pd.read_csv(ARQUIVO_EVENTOS_CLASSIFICADOS)

print("Distribuição original de estratos:")
display(df_google_raw['Estrato'].value_counts())

Carregando a base de eventos classificados...
Distribuição original de estratos:


Estrato
A3    171
A4    134
A1    110
B4     90
A2     86
B1     78
B2     60
B3     52
Name: count, dtype: int64

In [21]:
# A base de eventos usa rótulos B1-B4 para os estratos mais baixos; o projeto
# usa a faixa estendida A1-A8, então B1-B4 são remapeados para A5-A8.
print("Remapeando estratos B1-B4 -> A5-A8...")

mapeamento_estratos = {
    'B1': 'A5',
    'B2': 'A6',
    'B3': 'A7',
    'B4': 'A8'
}
df_google_raw['Estrato'] = df_google_raw['Estrato'].replace(mapeamento_estratos)

print("Nova distribuição de estratos:")
display(df_google_raw['Estrato'].value_counts())

# Padroniza o nome do evento em maiúsculas
df_google_raw['Nome do evento'] = df_google_raw['Nome do evento'].str.upper()

Remapeando estratos B1-B4 -> A5-A8...
Nova distribuição de estratos:


Estrato
A3    171
A4    134
A1    110
A8     90
A2     86
A5     78
A6     60
A7     52
Name: count, dtype: int64

### 10.2 Normalização de texto (remoção de acentos, maiúsculas, espaços)

In [22]:
print("Normalizando nomes de eventos (acentos, maiúsculas, espaços) em ambas as bases...")


def limpar_texto(serie):
    """Remove acentos, converte para maiúsculas, colapsa espaços múltiplos e tira espaços nas bordas."""
    return (serie.astype(str)
            .str.normalize('NFKD')
            .str.encode('ascii', errors='ignore')
            .str.decode('utf-8')
            .str.upper()
            .str.replace(r'\s+', ' ', regex=True)
            .str.strip())


df_base_congressos = df_congressos_unificado.copy()
df_base_congressos['evento_limpo'] = limpar_texto(df_base_congressos['titulo_evento_lattes'])
df_google_raw['Nome do evento'] = limpar_texto(df_google_raw['Nome do evento'])
df_google_raw['Nome do evento em inglês'] = limpar_texto(df_google_raw['Nome do evento em inglês'])
df_google_raw['Sigla'] = limpar_texto(df_google_raw['Sigla'])

# Para a busca por substring/sigla funcionar bem nos dois idiomas, ordenamos a
# base de eventos pelo maior nome disponível entre PT e EN. Isso evita que um
# nome curto "roube" o match de um nome mais longo e específico.
df_google_raw['tamanho_pt'] = df_google_raw['Nome do evento'].str.len()
df_google_raw['tamanho_en'] = df_google_raw['Nome do evento em inglês'].str.len()
df_google_raw['tamanho_max'] = df_google_raw[['tamanho_pt', 'tamanho_en']].max(axis=1)
df_google_raw = df_google_raw.sort_values(by='tamanho_max', ascending=False)

# Lista de dicionários — acesso mais rápido do que iterrows() em um DataFrame
lista_google = df_google_raw.to_dict('records')

print("Normalização concluída.")

Normalizando nomes de eventos (acentos, maiúsculas, espaços) em ambas as bases...
Normalização concluída.


### 10.3 Função de match fuzzy (sigla exata > similaridade textual PT/EN)

A função tenta, em ordem de confiança:

1. **Sigla exata**, isolada por limites de palavra (regex `\b`) — score 100.
2. **Similaridade fuzzy** (`token_set_ratio`) contra o nome do evento em
   português e em inglês, mantendo o maior score entre os dois.

Só é considerado match válido se o melhor score atingir o limiar de corte
(`LIMIAR_CORTE_FUZZY = 95`, em uma escala de 0 a 100).


In [23]:
LIMIAR_CORTE_FUZZY = 95  # Escala 0-100; valor alto para evitar falsos positivos


def encontrar_melhor_match_fuzzy(evento_lattes):
    """Procura, na base de eventos classificados, o melhor match fuzzy para um nome de evento.

    Retorna uma tupla: (sigla, nome_do_evento_padronizado, estrato, tipo_match, score_confianca).
    Quando não há match acima do limiar, retorna estrato 'A8' (pior classificação) e tipo 'Sem Match'.
    """
    if pd.isna(evento_lattes) or evento_lattes == 'NAN' or evento_lattes == '':
        return pd.NA, pd.NA, 'A8', 'Sem Match', 0

    melhor_google_match = None
    maior_score_encontrado = 0
    tipo_do_melhor_match = 'Sem Match'

    # --- Tentativa 1: sigla exata isolada por limites de palavra ---
    for google in lista_google:
        sigla = google['Sigla']
        if pd.notna(sigla) and sigla != 'NAN' and sigla != "":
            padrao = r'\b' + re.escape(sigla) + r'\b'
            if re.search(padrao, evento_lattes):
                return google['Sigla'], google['Nome do evento'], google['Estrato'], 'Por Sigla Exata', 100

    # --- Tentativa 2: similaridade fuzzy (token_set_ratio) em PT e EN ---
    for google in lista_google:
        nome_pt = google['Nome do evento']
        nome_en = google['Nome do evento em inglês']

        score_pt = 0
        score_en = 0

        if pd.notna(nome_pt) and nome_pt != 'NAN' and nome_pt != "":
            score_pt = fuzz.token_set_ratio(evento_lattes, nome_pt)

        if pd.notna(nome_en) and nome_en != 'NAN' and nome_en != "":
            score_en = fuzz.token_set_ratio(evento_lattes, nome_en)

        score_atual_max = max(score_pt, score_en)

        if score_atual_max > maior_score_encontrado:
            maior_score_encontrado = score_atual_max
            melhor_google_match = google
            tipo_do_melhor_match = 'Fuzzy Nome PT' if score_pt >= score_en else 'Fuzzy Nome EN'

    # --- Decisão final: o melhor score supera o limiar de segurança? ---
    if maior_score_encontrado >= LIMIAR_CORTE_FUZZY:
        return (
            melhor_google_match['Sigla'],
            melhor_google_match['Nome do evento'],
            melhor_google_match['Estrato'],
            tipo_do_melhor_match,
            maior_score_encontrado
        )

    # Score insuficiente: rejeita o match para evitar falso positivo
    return pd.NA, pd.NA, 'A8', 'Sem Match', maior_score_encontrado

### 10.4 Aplicação do match e consolidação de `df_congressos_unificado`

In [24]:
print("Aplicando o match fuzzy a todos os trabalhos de congresso (pode levar alguns segundos)...")

resultados = df_base_congressos['evento_limpo'].apply(encontrar_melhor_match_fuzzy)

df_base_congressos['sigla_evento_google'] = [res[0] for res in resultados]
df_base_congressos['titulo_evento_google'] = [res[1] for res in resultados]
df_base_congressos['estrato'] = [res[2] for res in resultados]
df_base_congressos['tipo_match'] = [res[3] for res in resultados]
df_base_congressos['score_confianca'] = [res[4] for res in resultados]

# --- Relatório-resumo do cruzamento ---
total_originais = len(df_base_congressos)
qtd_sigla = (df_base_congressos['tipo_match'] == 'Por Sigla Exata').sum()
qtd_fuzzy_pt = (df_base_congressos['tipo_match'] == 'Fuzzy Nome PT').sum()
qtd_fuzzy_en = (df_base_congressos['tipo_match'] == 'Fuzzy Nome EN').sum()
qtd_falhas = (df_base_congressos['tipo_match'] == 'Sem Match').sum()

print("\n--- Relatório de Cruzamento de Eventos ---")
print(f"Total de trabalhos de congresso (unificado): {total_originais}")
if total_originais:
    print(f"Match por Sigla Exata: {qtd_sigla} ({round((qtd_sigla/total_originais)*100, 1)}%)")
    print(f"Match Fuzzy (Nome PT):  {qtd_fuzzy_pt} ({round((qtd_fuzzy_pt/total_originais)*100, 1)}%)")
    print(f"Match Fuzzy (Nome EN):  {qtd_fuzzy_en} ({round((qtd_fuzzy_en/total_originais)*100, 1)}%)")
    print(f"Sem Match:              {qtd_falhas} ({round((qtd_falhas/total_originais)*100, 1)}%)")

    display(
        df_base_congressos[df_base_congressos['tipo_match'] != 'Sem Match']
        [['titulo_evento_lattes', 'titulo_evento_google', 'estrato', 'tipo_match', 'score_confianca']]
        .sample(min(5, total_originais))
    )

Aplicando o match fuzzy a todos os trabalhos de congresso (pode levar alguns segundos)...

--- Relatório de Cruzamento de Eventos ---
Total de trabalhos de congresso (unificado): 3958
Match por Sigla Exata: 1380 (34.9%)
Match Fuzzy (Nome PT):  110 (2.8%)
Match Fuzzy (Nome EN):  829 (20.9%)
Sem Match:              1639 (41.4%)


,titulo_evento_lattes,titulo_evento_google,estrato,tipo_match,score_confianca
330,LIII SIMPÓSIO BRASILEIRO DE PESQUISA OPERACION...,BRAZILIAN SYMPOSIUM ON OPERATIONS RESEARCH,A4,Por Sigla Exata,100.000000
3226,CF '20: COMPUTING FRONTIERS CONFERENCE,CONFERENCIA INTERNACIONAL ACM SOBRE FRONTEIRAS...,A3,Por Sigla Exata,100.000000
1863,2024 INTERNATIONAL JOINT CONFERENCE ON NEURAL ...,CONFERENCIA INTERNACIONAL CONJUNTA IEEE SOBRE ...,A1,Por Sigla Exata,100.000000
135,XXX SIMPÓSIO BRASILEIRO DE REDES DE COMPUTADORES,BRAZILIAN SYMPOSIUM ON COMPUTER NETWORKS AND D...,A4,Fuzzy Nome EN,95.348837
333,26TH INTERNATIONAL COMPUTING AND COMBINATORICS...,CONFERENCIA INTERNACIONAL DE COMPUTACAO E COMB...,A6,Por Sigla Exata,100.000000


### 10.5 Auditoria da "zona crítica" de confiança

Matches com score entre o limiar mínimo (95) e quase-perfeito (99) merecem
uma segunda olhada manual antes de confiar 100% no resultado.


In [25]:
limiar_inferior = 95
limiar_superior = 99

if not df_base_congressos.empty:
    df_zona_critica = df_base_congressos[
        (df_base_congressos['score_confianca'] >= limiar_inferior) &
        (df_base_congressos['score_confianca'] <= limiar_superior)
    ].copy()

    df_zona_critica = df_zona_critica.sort_values(by='score_confianca', ascending=True)

    colunas_para_auditoria = ['titulo_evento_lattes', 'titulo_evento_google', 'sigla_evento_google', 'estrato', 'tipo_match', 'score_confianca']
    tabela_auditoria = df_zona_critica[colunas_para_auditoria]

    print(f"Encontrados {len(tabela_auditoria)} registros na zona crítica (score {limiar_inferior} a {limiar_superior}).")
    print("Recomenda-se leitura atenta para garantir que não há homônimos.\n")

    display(
        tabela_auditoria.style.background_gradient(
            subset=['score_confianca'],
            cmap='YlOrRd_r',
            vmin=limiar_inferior,
            vmax=limiar_superior
        )
    )

Encontrados 133 registros na zona crítica (score 95 a 99).
Recomenda-se leitura atenta para garantir que não há homônimos.



,titulo_evento_lattes,titulo_evento_google,sigla_evento_google,estrato,tipo_match,score_confianca
3706,Proceedings of the 2011 5th Ftra International Conference on Multimedia and Ubiquitous Engineering Mue 2011,CONFERENCIA INTERNACIONAL ACM SOBRE MULTIMIDIA,ACMMM,A1,Fuzzy Nome EN,95.000000
3853,International Conference on Multimedia Computing and Systems Proceedings,CONFERENCIA INTERNACIONAL ACM SOBRE MULTIMIDIA,ACMMM,A1,Fuzzy Nome EN,95.000000
1152,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
499,XIX SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1159,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1160,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1161,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1150,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1141,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951
1422,XXI SIMPÓSIO BRASILEIRO DE TELECOMUNICAÇÕES,BRAZILIAN SYMPOSIUM ON TELECOMMUNICATIONS AND SIGNAL PROCESSING,SBRT,A8,Fuzzy Nome EN,95.121951


### 10.6 Consolidação final de `df_congressos_unificado`

In [26]:
COLUNAS_CONGRESSO_UNIFICADO = [c for c in COLUNAS_CONGRESSO if c != 'fonte'] + ['fontes', 'chave_dedup']

df_congressos_unificado = df_base_congressos[COLUNAS_CONGRESSO_UNIFICADO].copy()

print("Estrutura final de df_congressos_unificado:")
df_congressos_unificado.info()

Estrutura final de df_congressos_unificado:
<class 'pandas.DataFrame'>
RangeIndex: 3958 entries, 0 to 3957
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   id_lattes             3958 non-null   str   
 1   titulo_artigo         3958 non-null   str   
 2   ano                   3958 non-null   object
 3   doi                   1476 non-null   str   
 4   autores               3913 non-null   object
 5   titulo_evento_lattes  3908 non-null   object
 6   paginas               2853 non-null   object
 7   sigla_evento_google   2319 non-null   str   
 8   titulo_evento_google  2319 non-null   str   
 9   estrato               3958 non-null   str   
 10  tipo_match            3958 non-null   str   
 11  coautoria_aluno       0 non-null      object
 12  fontes                3958 non-null   str   
 13  chave_dedup           3958 non-null   object
dtypes: object(6), str(8)
memory usage: 1.0+ MB


## 11. Detecção de Coautoria de Alunos nas Produções

Mesma lógica da versão anterior deste notebook, aplicada a
`df_periodicos_unificado` e `df_congressos_unificado`. A estratégia
permanece: gerar exaustivamente todas as variações plausíveis do nome de
cada aluno e verificar se alguma delas aparece como substring isolada na
string de autores de cada publicação.

> A cobertura desta etapa depende de `autores` estar preenchido — para
> publicações de origem ORCID isso normalmente não acontece (ver
> limitação na Seção 6), então a coautoria de aluno tende a só ser
> detectada em publicações de origem Lattes ou Scopus.


### 11.1 Normalização de texto e geração de variações de nome

In [27]:
def normalizar_texto(valor):
    """Remove acentos, força maiúsculas e reduz qualquer caractere não alfanumérico a um único espaço."""
    if pd.isna(valor):
        return ""

    texto = unicodedata.normalize('NFKD', str(valor))
    texto = texto.encode('ascii', errors='ignore').decode('utf-8')
    texto = texto.upper()
    texto = re.sub(r'[^A-Z0-9]+', ' ', texto)
    return re.sub(r'\s+', ' ', texto).strip()


def gerar_todas_abreviacoes(nome_completo_norm):
    """Gera o conjunto de variações plausíveis de citação acadêmica de um nome completo normalizado.

    Cobre os padrões mais comuns na autoria brasileira: uso do sobrenome
    materno/intermediário como "sobrenome de citação", sobrenomes compostos,
    nomes de batismo duplos e a inversão Sobrenome-Nome / Nome-Sobrenome.
    """
    preposicoes = {'DE', 'DA', 'DO', 'DAS', 'DOS', 'E'}
    partes_originais = nome_completo_norm.split()
    partes_uteis = [p for p in partes_originais if p not in preposicoes]

    if len(partes_uteis) < 2:
        return {nome_completo_norm}

    variacoes = set([nome_completo_norm])

    # --- Define os possíveis blocos de "sobrenome de citação" ---
    sobrenomes_alvo = [partes_uteis[-1]]  # último nome (ex.: "CARNEIRO")

    # Qualquer nome do meio também pode ser o sobrenome de citação
    for i in range(1, len(partes_uteis) - 1):
        sobrenomes_alvo.append(partes_uteis[i])

    # Combinação composta dos dois últimos nomes (ex.: "DIAS CARNEIRO")
    if len(partes_uteis) >= 3:
        sobrenomes_alvo.append(f"{partes_uteis[-2]} {partes_uteis[-1]}")

    # Com a preposição original, se existir (ex.: "DE CARNEIRO")
    if len(partes_originais) >= 2 and partes_originais[-2] in preposicoes:
        sobrenomes_alvo.append(f"{partes_originais[-2]} {partes_originais[-1]}")

    # --- Combina cada sobrenome candidato com as iniciais do nome restante ---
    for sobrenome in set(sobrenomes_alvo):
        sobrenome_partes = sobrenome.split()
        resto = [p for p in partes_uteis if p not in sobrenome_partes]

        if not resto:
            continue

        iniciais = [p[0] for p in resto]
        primeiro_nome = resto[0]
        inicial_primeira = iniciais[0]

        iniciais_com_espaco = " ".join(iniciais)
        iniciais_sem_espaco = "".join(iniciais)
        duas_iniciais = f"{iniciais[0]} {iniciais[1]}" if len(iniciais) > 1 else inicial_primeira

        primeiro_mais_iniciais = primeiro_nome
        if len(iniciais) > 1:
            primeiro_mais_iniciais += " " + " ".join(iniciais[1:])

        # Possíveis blocos do "nome de batismo" (a parte antes do sobrenome)
        blocos_nome = [
            iniciais_com_espaco,       # ex.: "J V D C"
            iniciais_sem_espaco,       # ex.: "JVDC"
            inicial_primeira,          # ex.: "J"
            duas_iniciais,             # ex.: "J V"
            primeiro_nome,             # ex.: "JOAO"
            primeiro_mais_iniciais,    # ex.: "JOAO V D C"
            " ".join(resto)            # ex.: "JOAO VITOR DIAS CARNEIRO"
        ]

        # Permutação da ordem: Sobrenome-Nome e Nome-Sobrenome
        for bloco in set(blocos_nome):
            variacoes.add(f"{sobrenome} {bloco}")
            variacoes.add(f"{bloco} {sobrenome}")

    return variacoes

### 11.2 Extração dos alunos a partir dos JSONs brutos

In [28]:
def extrair_alunos(diretorio_alunos):
    """Lê os JSONs brutos dos alunos e monta um DataFrame com todas as variações de nome geradas."""
    registros = []
    pasta = Path(diretorio_alunos)

    if not pasta.exists() or not pasta.is_dir():
        print(f"AVISO: diretório de alunos não encontrado: {diretorio_alunos}")
        return pd.DataFrame()

    for arquivo_json in sorted(pasta.glob('*.json')):
        try:
            with open(arquivo_json, 'r', encoding='utf-8') as f:
                dados_aluno = json.load(f)

            info = dados_aluno.get('informacoes_pessoais', {})
            id_lattes = info.get('id_lattes')
            nome_completo = info.get('nome_completo', '')
            nome_citacoes = info.get('nome_citacoes', '')

            if not id_lattes or not nome_completo:
                continue

            # Variações de citação que o próprio Lattes do aluno já declara
            citacoes = [item.strip() for item in str(nome_citacoes).split(';') if item.strip()]

            nome_comp_norm = normalizar_texto(nome_completo)
            todas_permutacoes = gerar_todas_abreviacoes(nome_comp_norm)

            variacoes_normalizadas = list(todas_permutacoes)
            for variacao in citacoes:
                nome_norm_citacao = normalizar_texto(variacao)
                if nome_norm_citacao and nome_norm_citacao not in variacoes_normalizadas:
                    variacoes_normalizadas.append(nome_norm_citacao)

            registros.append({
                'id_lattes': str(id_lattes),
                'nome_completo': str(nome_completo).strip(),
                'nome_citacoes': str(nome_citacoes).strip(),
                'nome_completo_normalizado': nome_comp_norm,
                'nome_citacoes_normalizadas': ' | '.join(variacoes_normalizadas),
                'variacoes_coautoria': ' | '.join(variacoes_normalizadas),
            })
        except (json.JSONDecodeError, OSError) as erro:
            print(f"Erro ao ler {arquivo_json}: {erro}")

    df_alunos_local = pd.DataFrame(registros)
    if not df_alunos_local.empty:
        df_alunos_local = df_alunos_local.drop_duplicates(subset=['id_lattes']).copy()

    return df_alunos_local


print("Extraindo dados de alunos...")
df_alunos = extrair_alunos(CAMINHO_PASTA_ALUNOS)
print(f"Total de alunos carregados: {len(df_alunos)}")
display(df_alunos.head())

Extraindo dados de alunos...
Total de alunos carregados: 297


,id_lattes,nome_completo,nome_citacoes,nome_completo_normalizado,nome_citacoes_normalizadas,variacoes_coautoria
0,1766965412894981,João Luís da Silva Guio Soares,"SOARES, J. L. S. G.;GUIO, J. L.",JOAO LUIS DA SILVA GUIO SOARES,SILVA J L G S | J L SOARES | SILVA J L | LUIS ...,SILVA J L G S | J L SOARES | SILVA J L | LUIS ...
1,0352188533423371,David Ventura Cardoso,"CARDOSO, D. V.",DAVID VENTURA CARDOSO,VENTURA CARDOSO DAVID | DAVID CARDOSO | DC VEN...,VENTURA CARDOSO DAVID | DAVID CARDOSO | DC VEN...
2,8773283315440616,Ana Clara Correa da Silva,"SILVA, A. C. C.",ANA CLARA CORREA DA SILVA,SILVA ACC | ANA CLARA | A C CORREA | CORREA AN...,SILVA ACC | ANA CLARA | A C CORREA | CORREA AN...
3,6910314996365495,Fabio Luiz Silva Nogueira,"NOGUEIRA, F. L. S.",FABIO LUIZ SILVA NOGUEIRA,F L SILVA | FABIO L N SILVA | FLN SILVA | FABI...,F L SILVA | FABIO L N SILVA | FLN SILVA | FABI...
4,2636706873331793,Felipe Bevilaqua Foldes Guimarães,"GUIMARÃES, F. B. F.;GUIMARÃES, FELIPE BEVILAQU...",FELIPE BEVILAQUA FOLDES GUIMARAES,F FOLDES | FELIPE FOLDES GUIMARAES BEVILAQUA |...,F FOLDES | FELIPE FOLDES GUIMARAES BEVILAQUA |...


### 11.3 Verificação de coautoria nas duas tabelas unificadas

In [29]:
def montar_lista_variacoes(df_alunos_local):
    """Transforma a coluna 'variacoes_coautoria' de cada aluno em um conjunto de strings, agrupados por aluno."""
    variacoes = []
    if df_alunos_local.empty:
        return variacoes

    for _, linha in df_alunos_local.iterrows():
        nomes = [item.strip() for item in str(linha.get('variacoes_coautoria', '')).split('|') if item.strip()]
        if nomes:
            variacoes.append(set(nomes))

    return variacoes


def tem_coautoria_aluno(autores, variacoes_alunos):
    """Verifica se a string de autores de uma publicação contém alguma variação de nome de algum aluno.

    A comparação usa espaços como delimitadores nas duas pontas para evitar
    que uma variação curta (ex.: uma única inicial) seja encontrada como
    substring de outra palavra sem relação nenhuma.
    """
    if pd.isna(autores) or not str(autores).strip() or not variacoes_alunos:
        return False

    autores_normalizados = f" {normalizar_texto(autores)} "

    for variacoes in variacoes_alunos:
        for nome_normalizado in variacoes:
            if f" {nome_normalizado} " in autores_normalizados:
                return True

    return False


print("Marcando coautoria de alunos em df_periodicos_unificado e df_congressos_unificado...")
variacoes_alunos = montar_lista_variacoes(df_alunos)

for df_prod in [df_periodicos_unificado, df_congressos_unificado]:
    if 'autores' in df_prod.columns:
        df_prod['coautoria_aluno'] = df_prod['autores'].apply(
            lambda autores: tem_coautoria_aluno(autores, variacoes_alunos)
        )
    else:
        df_prod['coautoria_aluno'] = False

print(f"Periódicos com coautoria de aluno:   {int(df_periodicos_unificado['coautoria_aluno'].sum())}")
print(f"Conferências com coautoria de aluno: {int(df_congressos_unificado['coautoria_aluno'].sum())}")

Marcando coautoria de alunos em df_periodicos_unificado e df_congressos_unificado...
Periódicos com coautoria de aluno:   904
Conferências com coautoria de aluno: 1988


## 12. Propagação do Tratamento de Volta para as Tabelas por Fonte

As Seções 9-11 calcularam o enriquecimento (percentil, estrato, match,
coautoria de aluno) **uma única vez**, sobre a base já deduplicada. Esta
seção devolve esse resultado para `df_periodicos_bruto`/`df_congressos_bruto`
(que ainda têm uma linha por publicação **por fonte**), usando `chave_dedup`
como elo — o mesmo papel que uma FK exerceria, só que resolvido em pandas
antes do `INSERT`, o que evita ter que rodar `UPDATE`s no DuckDB depois.

Junto com o enriquecimento vão também duas colunas que existem justamente
para viabilizar os relatórios de lacuna:

- **`chave_dedup`** — o identificador da publicação dentro da lista daquele
  professor, idêntico nas 7 tabelas. É por ele que se cruza a tabela
  unificada com qualquer tabela por fonte.
- **`fontes`** — a lista das bases em que aquela publicação foi encontrada
  (ex.: `LATTES,SCOPUS`), já refletindo o resultado da deduplicação.

O resultado é dividido de volta nas três fontes — essas seis variáveis
(`df_artigos_periodico_lattes/orcid/scopus`,
`df_artigos_congresso_lattes/orcid/scopus`) populam as 6 tabelas por fonte
na Seção 13. `montar_relatorio_lacunas` abre a coluna `fontes` em três
booleanos (`em_lattes`/`em_orcid`/`em_scopus`) para conferência rápida aqui
no notebook; no banco, a consulta equivalente é:

```sql
-- artigos de periódico que o professor tem, mas que faltam no ORCID
SELECT u.id_lattes, u.titulo_artigo, u.doi, u.fontes
FROM tb_artigo_periodico u
LEFT JOIN tb_artigo_periodico_orcid o ON o.chave_dedup = u.chave_dedup
WHERE o.chave_dedup IS NULL;
```

In [30]:
COLUNAS_ENRIQUECIMENTO_PERIODICO = [
    'match_adequado', 'id_scopus', 'titulo_revista_scopus', 'maior_percentil',
    'codigo_area_maior_percentil', 'area_maior_percentil', 'issn',
    'computation_area', 'coautoria_aluno',
]

COLUNAS_ENRIQUECIMENTO_CONGRESSO = [
    'sigla_evento_google', 'titulo_evento_google', 'estrato', 'tipo_match', 'coautoria_aluno',
]

# Além do enriquecimento, cada linha por fonte recebe também `fontes` (todas
# as bases em que aquela publicação foi encontrada) e `chave_dedup` (o elo
# com a tabela unificada). São essas duas colunas que permitem responder
# "quais artigos do professor X estão no Lattes mas faltam no ORCID?".
COLUNAS_PERIODICO_POR_FONTE = COLUNAS_PERIODICO + ['fontes', 'chave_dedup']
COLUNAS_CONGRESSO_POR_FONTE = COLUNAS_CONGRESSO + ['fontes', 'chave_dedup']


def propagar_enriquecimento(df_bruto, df_unificado_tratado, colunas_enriquecimento):
    """Devolve para a base bruta (uma linha por publicação por fonte) os
    valores já tratados na base unificada, ligando as duas por `chave_dedup`.
    Além das colunas de cruzamento, traz `fontes`, para que cada linha por
    fonte saiba em quais outras bases aquela publicação também aparece."""
    colunas_a_trazer = colunas_enriquecimento + ['fontes']
    df_enriquecimento = df_unificado_tratado[['chave_dedup'] + colunas_a_trazer]
    return (
        df_bruto.drop(columns=colunas_a_trazer, errors='ignore')
        .merge(df_enriquecimento, on='chave_dedup', how='left')
    )


print("Propagando os valores tratados de volta para as linhas por fonte...")
df_periodicos_bruto = propagar_enriquecimento(df_periodicos_bruto, df_periodicos_unificado, COLUNAS_ENRIQUECIMENTO_PERIODICO)
df_congressos_bruto = propagar_enriquecimento(df_congressos_bruto, df_congressos_unificado, COLUNAS_ENRIQUECIMENTO_CONGRESSO)


def dividir_por_fonte(df_bruto, colunas_finais, fonte):
    """Isola as linhas de uma fonte específica, já no schema final."""
    return df_bruto[df_bruto['fonte'] == fonte][colunas_finais].reset_index(drop=True)


df_artigos_periodico_lattes = dividir_por_fonte(df_periodicos_bruto, COLUNAS_PERIODICO_POR_FONTE, 'LATTES')
df_artigos_periodico_orcid = dividir_por_fonte(df_periodicos_bruto, COLUNAS_PERIODICO_POR_FONTE, 'ORCID')
df_artigos_periodico_scopus = dividir_por_fonte(df_periodicos_bruto, COLUNAS_PERIODICO_POR_FONTE, 'SCOPUS')

df_artigos_congresso_lattes = dividir_por_fonte(df_congressos_bruto, COLUNAS_CONGRESSO_POR_FONTE, 'LATTES')
df_artigos_congresso_orcid = dividir_por_fonte(df_congressos_bruto, COLUNAS_CONGRESSO_POR_FONTE, 'ORCID')
df_artigos_congresso_scopus = dividir_por_fonte(df_congressos_bruto, COLUNAS_CONGRESSO_POR_FONTE, 'SCOPUS')


def montar_relatorio_lacunas(df_unificado, rotulo):
    """Monta a visão de lacunas: uma linha por publicação de cada professor,
    com uma coluna booleana por base indicando presença/ausência. É a mesma
    informação que a coluna `fontes` carrega, só que já aberta para conferência
    rápida aqui no notebook (no banco, a consulta equivalente cruza
    `tb_artigo_*` com `tb_artigo_*_<fonte>` por `chave_dedup`)."""
    if df_unificado.empty:
        return pd.DataFrame(columns=['id_lattes', 'titulo_artigo', 'chave_dedup',
                                     'em_lattes', 'em_orcid', 'em_scopus'])
    relatorio = df_unificado[['id_lattes', 'titulo_artigo', 'doi', 'chave_dedup', 'fontes']].copy()
    for fonte in ['LATTES', 'ORCID', 'SCOPUS']:
        relatorio[f'em_{fonte.lower()}'] = relatorio['fontes'].fillna('').str.split(',').apply(lambda lista: fonte in lista)
    faltando = (~relatorio[['em_lattes', 'em_orcid', 'em_scopus']]).sum()
    print(f"  {rotulo}: {len(relatorio)} publicações | ausentes no Lattes: {faltando['em_lattes']} | "
          f"no ORCID: {faltando['em_orcid']} | no Scopus: {faltando['em_scopus']}")
    return relatorio


print("\nPropagação concluída. Volumes finais por fonte:")
print(f"  Periódico  -- LATTES: {len(df_artigos_periodico_lattes)} | ORCID: {len(df_artigos_periodico_orcid)} | SCOPUS: {len(df_artigos_periodico_scopus)}")
print(f"  Congresso  -- LATTES: {len(df_artigos_congresso_lattes)} | ORCID: {len(df_artigos_congresso_orcid)} | SCOPUS: {len(df_artigos_congresso_scopus)}")
print(f"  Unificado (deduplicado) -- Periódico: {len(df_periodicos_unificado)} | Congresso: {len(df_congressos_unificado)}")

print("\nLacunas por base:")
df_lacunas_periodicos = montar_relatorio_lacunas(df_periodicos_unificado, 'Periódicos')
df_lacunas_congressos = montar_relatorio_lacunas(df_congressos_unificado, 'Congressos')
display(df_lacunas_periodicos.head())

Propagando os valores tratados de volta para as linhas por fonte...

Propagação concluída. Volumes finais por fonte:
  Periódico  -- LATTES: 1997 | ORCID: 783 | SCOPUS: 1306
  Congresso  -- LATTES: 3639 | ORCID: 657 | SCOPUS: 1130
  Unificado (deduplicado) -- Periódico: 2311 | Congresso: 3958

Lacunas por base:
  Periódicos: 2311 publicações | ausentes no Lattes: 314 | no ORCID: 1528 | no Scopus: 1005
  Congressos: 3958 publicações | ausentes no Lattes: 319 | no ORCID: 3301 | no Scopus: 2828


,id_lattes,titulo_artigo,doi,chave_dedup,fontes,em_lattes,em_orcid,em_scopus
0,0211300683784278,On the (In)Dependence of the Peano Axioms for ...,http://dx.doi.org/10.1080/01445340.2021.1971005,0211300683784278|DOI:10.1080/01445340.2021.197...,"LATTES,ORCID,SCOPUS",True,True,True
1,0211300683784278,Short proofs on the structure of general parti...,http://dx.doi.org/10.1016/j.dam.2020.09.007,0211300683784278|DOI:10.1016/j.dam.2020.09.007,"LATTES,SCOPUS",True,False,True
2,0211300683784278,Transversals of longest paths,http://dx.doi.org/10.1016/j.disc.2019.111717,0211300683784278|DOI:10.1016/j.disc.2019.111717,"LATTES,SCOPUS",True,False,True
3,0211300683784278,Intersection of longest paths in graph classes,http://dx.doi.org/10.1016/j.dam.2019.03.022,0211300683784278|DOI:10.1016/j.dam.2019.03.022,"LATTES,SCOPUS",True,False,True
4,0211300683784278,"L(2,1)-labelling of graphs with few P4?s",10.1016/j.disopt.2016.01.006,0211300683784278|DOI:10.1016/j.disopt.2016.01.006,"LATTES,SCOPUS",True,False,True


## 13. Persistência Consolidada no DuckDB

Cria (se não existir) o schema relacional em `pesquisadores_teste.duckdb` e
carrega os DataFrames tratados. **Onze tabelas** ao todo:

- `tb_professores` — tabela "mãe", uma linha por professor (chave
  `id_lattes`); agora com `orcid_id`/`scopus_author_id` (nullable).
- `tb_alunos` — uma linha por aluno, com as variações de nome da Seção 11.
- `tb_artigo_periodico` / `tb_artigo_conferencia` — **mesmo formato de
  antes** (é o que `app.py` consulta), agora com uma coluna `fontes`
  adicional; populadas a partir da base unificada e deduplicada.
- `tb_orientacoes` — uma orientação por linha.
- **Seis tabelas novas**, uma por fonte × tipo de produção
  (`tb_artigo_periodico_lattes/orcid/scopus`,
  `tb_artigo_conferencia_lattes/orcid/scopus`) — mesmo schema das tabelas
  unificadas (mais uma coluna `fonte` de valor único), preservando uma
  linha por publicação por fonte, já com os campos de cruzamento
  propagados na Seção 12.

Todas as tabelas de artigos têm `FOREIGN KEY (id_lattes) REFERENCES
tb_professores(id_lattes)` — `id_lattes` continua sendo a única chave
primária/estrangeira do modelo; `orcid_id` e `scopus_author_id` vivem como
atributos de `tb_professores`, não como chaves estrangeiras separadas.

A carga é feita em modo *replace*: as tabelas filhas são limpas antes da
tabela mãe (para não violar a integridade referencial) e, em seguida, todo o
conteúdo tratado em memória é inserido novamente.


### 13.1 Criação do schema (tabelas, sequências e chaves estrangeiras)

In [31]:
print("Conectando ao DuckDB e criando o schema (se ainda não existir)...")
con = duckdb.connect(ARQUIVO_DUCKDB_DESTINO)

# --- Tabela mãe: Professores ---
con.execute("""
CREATE TABLE IF NOT EXISTS tb_professores (
    id_lattes VARCHAR PRIMARY KEY,
    nome_completo VARCHAR,
    nome_citacoes VARCHAR,
    sexo VARCHAR,
    rotulo VARCHAR,
    periodo VARCHAR,
    bolsa_produtividade VARCHAR,
    endereco_profissional VARCHAR,
    atualizacao_cv TIMESTAMP,
    url VARCHAR,
    texto_resumo VARCHAR,
    orcid_id VARCHAR,
    scopus_author_id VARCHAR
);
""")

# --- Tabela de Alunos ---
con.execute("""
CREATE TABLE IF NOT EXISTS tb_alunos (
    id_lattes VARCHAR PRIMARY KEY,
    nome_completo VARCHAR,
    nome_citacoes VARCHAR,
    nome_completo_normalizado VARCHAR,
    nome_citacoes_normalizadas VARCHAR,
    variacoes_coautoria VARCHAR
);
""")

# --- Sequências para os IDs automáticos das tabelas filhas ---
for nome_sequencia in [
    'seq_id_artigo_periodico', 'seq_id_artigo_conferencia', 'seq_id_orientacao',
    'seq_id_artigo_periodico_lattes', 'seq_id_artigo_periodico_orcid', 'seq_id_artigo_periodico_scopus',
    'seq_id_artigo_conferencia_lattes', 'seq_id_artigo_conferencia_orcid', 'seq_id_artigo_conferencia_scopus',
]:
    con.execute(f"CREATE SEQUENCE IF NOT EXISTS {nome_sequencia};")

# --- Tabela Filha: Artigos de Periódico (unificada, cruzada com Scopus) ---
con.execute("""
CREATE TABLE IF NOT EXISTS tb_artigo_periodico (
    id_artigo_periodico INTEGER PRIMARY KEY DEFAULT nextval('seq_id_artigo_periodico'),
    id_lattes VARCHAR,
    titulo_artigo VARCHAR NOT NULL,
    titulo_revista_lattes VARCHAR,
    ano_pub INTEGER,
    doi VARCHAR,
    autores VARCHAR,
    match_adequado BOOLEAN,
    coautoria_aluno BOOLEAN,
    id_scopus VARCHAR,
    titulo_revista_scopus VARCHAR,
    maior_percentil INTEGER,
    codigo_area_maior_percentil VARCHAR,
    area_maior_percentil VARCHAR,
    issn VARCHAR,
    computation_area BOOLEAN,
    fontes VARCHAR,
    chave_dedup VARCHAR,
    FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
);
""")

# --- Tabela Filha: Artigos de Conferência (unificada, cruzada com a base de eventos) ---
con.execute("""
CREATE TABLE IF NOT EXISTS tb_artigo_conferencia (
    id_artigo_conferencia INTEGER PRIMARY KEY DEFAULT nextval('seq_id_artigo_conferencia'),
    id_lattes VARCHAR,
    titulo_artigo VARCHAR NOT NULL,
    ano INTEGER,
    doi VARCHAR,
    autores VARCHAR,
    titulo_evento_lattes VARCHAR,
    paginas VARCHAR,
    sigla_evento_google VARCHAR,
    titulo_evento_google VARCHAR,
    estrato VARCHAR,
    tipo_match VARCHAR,
    coautoria_aluno BOOLEAN,
    fontes VARCHAR,
    chave_dedup VARCHAR,
    FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
);
""")

# --- Tabela Filha: Orientações ---
con.execute("""
CREATE TABLE IF NOT EXISTS tb_orientacoes (
    id_orientacao INTEGER PRIMARY KEY DEFAULT nextval('seq_id_orientacao'),
    id_lattes VARCHAR,
    titulo_trabalho VARCHAR,
    ano_inicio INTEGER,
    orientando VARCHAR,
    tipo_trabalho VARCHAR,
    instituicao VARCHAR,
    curso VARCHAR,
    status VARCHAR,
    nivel VARCHAR,
    ano_conclusao INTEGER,
    FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
);
""")

# --- Seis tabelas novas: uma por fonte x tipo de produção (dado pré-deduplicação) ---
COLUNAS_SQL_ARTIGO_PERIODICO_FONTE = """
    id_lattes VARCHAR,
    titulo_artigo VARCHAR NOT NULL,
    titulo_revista_lattes VARCHAR,
    ano_pub INTEGER,
    doi VARCHAR,
    autores VARCHAR,
    match_adequado BOOLEAN,
    coautoria_aluno BOOLEAN,
    id_scopus VARCHAR,
    titulo_revista_scopus VARCHAR,
    maior_percentil INTEGER,
    codigo_area_maior_percentil VARCHAR,
    area_maior_percentil VARCHAR,
    issn VARCHAR,
    computation_area BOOLEAN,
    fonte VARCHAR,
    fontes VARCHAR,
    chave_dedup VARCHAR,
"""

COLUNAS_SQL_ARTIGO_CONFERENCIA_FONTE = """
    id_lattes VARCHAR,
    titulo_artigo VARCHAR NOT NULL,
    ano INTEGER,
    doi VARCHAR,
    autores VARCHAR,
    titulo_evento_lattes VARCHAR,
    paginas VARCHAR,
    sigla_evento_google VARCHAR,
    titulo_evento_google VARCHAR,
    estrato VARCHAR,
    tipo_match VARCHAR,
    coautoria_aluno BOOLEAN,
    fonte VARCHAR,
    fontes VARCHAR,
    chave_dedup VARCHAR,
"""


def criar_tabela_por_fonte(nome_tabela, nome_sequencia, nome_pk, colunas_sql):
    con.execute(f"""
        CREATE TABLE IF NOT EXISTS {nome_tabela} (
            {nome_pk} INTEGER PRIMARY KEY DEFAULT nextval('{nome_sequencia}'),
            {colunas_sql}
            FOREIGN KEY (id_lattes) REFERENCES tb_professores(id_lattes)
        );
    """)


for fonte_sufixo in ['lattes', 'orcid', 'scopus']:
    criar_tabela_por_fonte(
        f'tb_artigo_periodico_{fonte_sufixo}', f'seq_id_artigo_periodico_{fonte_sufixo}',
        f'id_artigo_periodico_{fonte_sufixo}', COLUNAS_SQL_ARTIGO_PERIODICO_FONTE,
    )
    criar_tabela_por_fonte(
        f'tb_artigo_conferencia_{fonte_sufixo}', f'seq_id_artigo_conferencia_{fonte_sufixo}',
        f'id_artigo_conferencia_{fonte_sufixo}', COLUNAS_SQL_ARTIGO_CONFERENCIA_FONTE,
    )

# Garante que bancos criados em uma versão anterior do schema (sem estas
# colunas) sejam atualizados ao reexecutar o notebook. Cada ALTER é
# protegido por try/except porque o DuckDB ainda não suporta
# "ADD COLUMN IF NOT EXISTS" de forma totalmente idempotente em todas as versões.
for alter_sql in [
    "ALTER TABLE tb_artigo_periodico ADD COLUMN autores VARCHAR",
    "ALTER TABLE tb_artigo_periodico ADD COLUMN doi VARCHAR",
    "ALTER TABLE tb_artigo_periodico ADD COLUMN coautoria_aluno BOOLEAN",
    "ALTER TABLE tb_artigo_periodico ADD COLUMN fontes VARCHAR",
    "ALTER TABLE tb_artigo_conferencia ADD COLUMN coautoria_aluno BOOLEAN",
    "ALTER TABLE tb_artigo_conferencia ADD COLUMN fontes VARCHAR",
    "ALTER TABLE tb_artigo_periodico ADD COLUMN chave_dedup VARCHAR",
    "ALTER TABLE tb_artigo_conferencia ADD COLUMN chave_dedup VARCHAR",
] + [
    f"ALTER TABLE tb_artigo_{tipo}_{fonte} ADD COLUMN {coluna} VARCHAR"
    for tipo in ['periodico', 'conferencia']
    for fonte in ['lattes', 'orcid', 'scopus']
    for coluna in ['fontes', 'chave_dedup']
] + [
    "ALTER TABLE tb_alunos ADD COLUMN nome_completo_normalizado VARCHAR",
    "ALTER TABLE tb_alunos ADD COLUMN nome_citacoes_normalizadas VARCHAR",
    "ALTER TABLE tb_alunos ADD COLUMN variacoes_coautoria VARCHAR",
    "ALTER TABLE tb_professores ADD COLUMN orcid_id VARCHAR",
    "ALTER TABLE tb_professores ADD COLUMN scopus_author_id VARCHAR",
]:
    try:
        con.execute(alter_sql)
    except Exception:
        pass  # Coluna já existe — nada a fazer

print("Schema pronto: 5 tabelas principais + 6 tabelas por fonte (criadas ou já existentes).")

Conectando ao DuckDB e criando o schema (se ainda não existir)...
Schema pronto: 5 tabelas principais + 6 tabelas por fonte (criadas ou já existentes).


### 13.2 Carga dos dados (limpeza das tabelas antigas + inserção)

In [32]:
print("Limpando dados antigos antes da nova carga (filhas primeiro, mãe depois)...")
# A ordem importa: as tabelas filhas têm FOREIGN KEY para tb_professores,
# então precisam ser esvaziadas antes da tabela mãe.
tabelas_filhas = [
    'tb_artigo_periodico', 'tb_artigo_conferencia', 'tb_orientacoes',
    'tb_artigo_periodico_lattes', 'tb_artigo_periodico_orcid', 'tb_artigo_periodico_scopus',
    'tb_artigo_conferencia_lattes', 'tb_artigo_conferencia_orcid', 'tb_artigo_conferencia_scopus',
]
for tabela in tabelas_filhas:
    con.execute(f"DELETE FROM {tabela}")
con.execute("DELETE FROM tb_alunos")
con.execute("DELETE FROM tb_professores")

print("Inserindo os dados tratados...")

# --- Tabela Mãe: Professores ---
if not df_pessoas.empty:
    con.execute("""
        INSERT INTO tb_professores (
            id_lattes, nome_completo, nome_citacoes, sexo, rotulo, periodo,
            bolsa_produtividade, endereco_profissional, atualizacao_cv, url,
            texto_resumo, orcid_id, scopus_author_id
        )
        SELECT
            id_lattes, nome_completo, nome_citacoes, sexo, rotulo, periodo,
            bolsa_produtividade, endereco_profissional, atualizacao_cv, url,
            texto_resumo, orcid_id, scopus_author_id
        FROM df_pessoas
    """)

# --- Tabela de Alunos ---
if not df_alunos.empty:
    con.execute("""
        INSERT INTO tb_alunos (
            id_lattes, nome_completo, nome_citacoes,
            nome_completo_normalizado, nome_citacoes_normalizadas, variacoes_coautoria
        )
        SELECT
            id_lattes, nome_completo, nome_citacoes,
            nome_completo_normalizado, nome_citacoes_normalizadas, variacoes_coautoria
        FROM df_alunos
    """)

# --- Tabela Unificada: Artigos de Periódico ---
if not df_periodicos_unificado.empty:
    con.execute("""
        INSERT INTO tb_artigo_periodico (
            id_lattes, titulo_artigo, titulo_revista_lattes, ano_pub,
            doi, autores, match_adequado, coautoria_aluno, id_scopus,
            titulo_revista_scopus, maior_percentil, codigo_area_maior_percentil,
            area_maior_percentil, issn, computation_area, fontes, chave_dedup
        )
        SELECT
            id_lattes, titulo_artigo, titulo_revista_lattes, ano_pub,
            doi, autores, match_adequado, coautoria_aluno, id_scopus,
            titulo_revista_scopus, maior_percentil, codigo_area_maior_percentil,
            area_maior_percentil, issn, computation_area, fontes, chave_dedup
        FROM df_periodicos_unificado
    """)

# --- Tabela Unificada: Artigos de Conferência ---
if not df_congressos_unificado.empty:
    con.execute("""
        INSERT INTO tb_artigo_conferencia (
            id_lattes, titulo_artigo, ano, doi, autores,
            titulo_evento_lattes, paginas, sigla_evento_google,
            titulo_evento_google, estrato, tipo_match, coautoria_aluno, fontes, chave_dedup
        )
        SELECT
            id_lattes, titulo_artigo, ano, doi, autores,
            titulo_evento_lattes, paginas, sigla_evento_google,
            titulo_evento_google, estrato, tipo_match, coautoria_aluno, fontes, chave_dedup
        FROM df_congressos_unificado
    """)

# --- Tabela Filha: Orientações ---
if not df_orientacoes.empty:
    con.execute("""
        INSERT INTO tb_orientacoes (
            id_lattes, titulo_trabalho, ano_inicio, orientando,
            tipo_trabalho, instituicao, curso, status, nivel, ano_conclusao
        )
        SELECT
            id_lattes, titulo_trabalho, ano_inicio, orientando,
            tipo_trabalho, instituicao, curso, status, nivel, ano_conclusao
        FROM df_orientacoes
    """)

# --- Seis tabelas por fonte (dado pré-deduplicação, já com o tratamento propagado) ---
mapa_tabelas_periodico_fonte = {
    'tb_artigo_periodico_lattes': df_artigos_periodico_lattes,
    'tb_artigo_periodico_orcid': df_artigos_periodico_orcid,
    'tb_artigo_periodico_scopus': df_artigos_periodico_scopus,
}
for nome_tabela, df_fonte in mapa_tabelas_periodico_fonte.items():
    if df_fonte.empty:
        continue
    con.register('df_fonte_periodico_tmp', df_fonte)
    con.execute(f"""
        INSERT INTO {nome_tabela} (
            id_lattes, titulo_artigo, titulo_revista_lattes, ano_pub,
            doi, autores, match_adequado, coautoria_aluno, id_scopus,
            titulo_revista_scopus, maior_percentil, codigo_area_maior_percentil,
            area_maior_percentil, issn, computation_area, fonte, fontes, chave_dedup
        )
        SELECT
            id_lattes, titulo_artigo, titulo_revista_lattes, ano_pub,
            doi, autores, match_adequado, coautoria_aluno, id_scopus,
            titulo_revista_scopus, maior_percentil, codigo_area_maior_percentil,
            area_maior_percentil, issn, computation_area, fonte, fontes, chave_dedup
        FROM df_fonte_periodico_tmp
    """)
    con.unregister('df_fonte_periodico_tmp')

mapa_tabelas_congresso_fonte = {
    'tb_artigo_conferencia_lattes': df_artigos_congresso_lattes,
    'tb_artigo_conferencia_orcid': df_artigos_congresso_orcid,
    'tb_artigo_conferencia_scopus': df_artigos_congresso_scopus,
}
for nome_tabela, df_fonte in mapa_tabelas_congresso_fonte.items():
    if df_fonte.empty:
        continue
    con.register('df_fonte_congresso_tmp', df_fonte)
    con.execute(f"""
        INSERT INTO {nome_tabela} (
            id_lattes, titulo_artigo, ano, doi, autores,
            titulo_evento_lattes, paginas, sigla_evento_google,
            titulo_evento_google, estrato, tipo_match, coautoria_aluno, fonte, fontes, chave_dedup
        )
        SELECT
            id_lattes, titulo_artigo, ano, doi, autores,
            titulo_evento_lattes, paginas, sigla_evento_google,
            titulo_evento_google, estrato, tipo_match, coautoria_aluno, fonte, fontes, chave_dedup
        FROM df_fonte_congresso_tmp
    """)
    con.unregister('df_fonte_congresso_tmp')

con.close()

print(f"Processo finalizado! Banco '{ARQUIVO_DUCKDB_DESTINO}' atualizado com o schema completo (11 tabelas).")

Limpando dados antigos antes da nova carga (filhas primeiro, mãe depois)...
Inserindo os dados tratados...
Processo finalizado! Banco 'pesquisadores_teste.duckdb' atualizado com o schema completo (11 tabelas).
